<!-- Cell 00 (Markdown) -->
<div dir=RTL>

# 📸 מערכת יצירת אלבומים דיגיטליים

**אירועי בת מצווה | תמונות מ-Google Drive → אלבום IDML מוכן להדפסה**

---

## 📋 תהליך מלא

1. **🔧 הגדרות** - התקנות וקונפיגורציה
2. **📐 פונקציות ליבה** - IDML Parser, Composer, עיבוד תמונות, LLM
3. **🔐 Google Drive API** - חיבור OAuth 2.0
4. **📥 הורדת תמונות** - מהדרייב לסביבת העבודה
5. **🤖 תיוג סצנות LLM Vision** - הבנה סמנטית של כל תמונה
6. **🚀 הצינור המלא** - תכנון, בחירה, יצירת IDML
7. **📤 העלאה חזרה לדרייב** - האלבום הסופי
8. **🧪 ניסויים** - כיול פרמטרים

## 🎯 חשוב לדעת

מחברת זו מיועדת ל-**Google Colab** עם גישה ל-**Google Drive**.

**לפני הרצה:**
- וודא שאתה ב-Colab (Runtime → Change runtime type)
- מומלץ: GPU T4 (חינמי) - מאיץ עיבוד תמונות
- צריך: Google Cloud Project עם Drive API פעיל ו-`client_secrets.json`

</div>

<!-- Cell 01 (Markdown) -->
<div dir=RTL>

## 🔧 שלב 0: התקנות והגדרות

הרץ פעם אחת את התא הבא להתקנת כל הספריות הנדרשות.

</div>

In [ ]:
# === Cell 02 ===
# התקנת ספריות נדרשות
import subprocess
import sys

packages = [
    'Pillow',
    'opencv-python',
    'opencv-contrib-python',
    'numpy',
    'requests',
    # Google Drive API
    'google-api-python-client',
    'google-auth-httplib2',
    'google-auth-oauthlib',
]

print("מתקין ספריות...\n")
for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        print(f"  ✓ {package}")
    except Exception as e:
        print(f"  ⚠️ {package} - {e}")

print("\n✅ התקנות הושלמו")

In [ ]:
# === Cell 03 ===
# Imports - כל מה שצריך
import os
import sys
import json
import math
import time
import uuid
import shutil
import zipfile
import hashlib
import io
import sqlite3
import base64
import xml.etree.ElementTree as ET
import glob
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Any
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from itertools import permutations
from abc import ABC, abstractmethod

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageEnhance
import requests

# הגדרות בסיסיות
PROJECT_DIR = Path.cwd() / "album_workspace"
PROJECT_DIR.mkdir(exist_ok=True)

DOWNLOADED_IMAGES_DIR = PROJECT_DIR / "downloaded_images"
DOWNLOADED_IMAGES_DIR.mkdir(exist_ok=True)

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# הגדרות האלבום
CM_TO_PT = 28.3465
ALBUM_CONFIG = {
    'total_pages': 16,
    'images_per_page': 6,
    'page_width_cm': 60,
    'page_height_cm': 30,
    'target_dpi': 300,
    'bleed_mm': 3,
}

LLM_CONFIG = {
    'provider': 'auto',  # auto / gemini / claude / simulated
    'temperature': 0.4,
}

# הגדרות תיוג סצנות (LLM Vision)
SCENE_TAGGING_CONFIG = {
    'batch_size': 5,            # כמה תמונות לשלוח ל-LLM בבת אחת
    'max_image_size_kb': 200,   # גודל מקסימלי לתמונה לפני שליחה (חוסך טוקנים)
    'cache_results': True,      # שומר תוצאות לקובץ JSON כדי לא לשלם פעמיים
}

print(f"📁 תיקיית עבודה: {PROJECT_DIR}")
print(f"⚙️  הגדרות אלבום: {ALBUM_CONFIG}")

<!-- Cell 04 (Markdown) -->
<div dir=RTL>

## 📐 שלב 1: IDML Parser

קוד שקורא קובץ IDML <br>
(שהוא בעצם ZIP עם XML בפנים) <br>
ומוציא מידע מובנה על ה-slots

In [3]:
# === Cell 05 ===
@dataclass
class SlotInfo:
    """מידע על slot בתבנית"""
    slot_id: str
    name: str
    center_x_cm: float
    center_y_cm: float
    width_cm: float
    height_cm: float
    rotation_deg: float
    aspect_ratio: float
    aspect_category: str  # 'landscape', 'portrait', 'square'
    area_cm2: float


@dataclass
class TemplateInfo:
    """מידע על התבנית כולה"""
    page_width_cm: float
    page_height_cm: float
    num_slots: int
    slots: List[SlotInfo]

    def to_dict(self):
        return {
            'page_width_cm': self.page_width_cm,
            'page_height_cm': self.page_height_cm,
            'num_slots': self.num_slots,
            'slots': [asdict(s) for s in self.slots]
        }


def parse_item_transform(transform_str: str):
    """פירוק מחרוזת ItemTransform של IDML: a b c d tx ty"""
    parts = transform_str.split()
    if len(parts) != 6:
        raise ValueError(f"Invalid ItemTransform: {transform_str}")
    a, b, c, d, tx, ty = [float(p) for p in parts]
    rotation_deg = math.degrees(math.atan2(b, a))
    return {'tx': tx, 'ty': ty, 'rotation_deg': rotation_deg}


def parse_path_geometry(rect_elem, ns):
    """קריאת הגודל מ-PathGeometry של Rectangle"""
    path_points = rect_elem.findall('.//PathPointType')
    if not path_points:
        return None, None

    xs, ys = [], []
    for pp in path_points:
        anchor = pp.get('Anchor', '')
        if anchor:
            x, y = [float(v) for v in anchor.split()]
            xs.append(x)
            ys.append(y)

    if not xs:
        return None, None

    return max(xs) - min(xs), max(ys) - min(ys)


def classify_aspect(ratio: float) -> str:
    if ratio > 1.2:
        return "landscape"
    elif ratio < 0.83:
        return "portrait"
    return "square"


def parse_idml(idml_path: str) -> TemplateInfo:
    """Parser ראשי ל-IDML"""
    with zipfile.ZipFile(idml_path, 'r') as z:
        # גודל דף
        prefs_xml = z.read('Resources/Preferences.xml').decode('utf-8')
        ns = {'idPkg': 'http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging'}
        root = ET.fromstring(prefs_xml)
        doc_pref = root.find('.//DocumentPreference')
        page_width_pt = float(doc_pref.get('PageWidth', 0))
        page_height_pt = float(doc_pref.get('PageHeight', 0))

        # slots מה-Spreads
        spread_files = [n for n in z.namelist() if n.startswith('Spreads/Spread_')]
        all_slots = []
        slot_index = 0

        for spread_file in spread_files:
            spread_xml = z.read(spread_file).decode('utf-8')
            spread_root = ET.fromstring(spread_xml)
            rectangles = spread_root.findall('.//Rectangle')

            for rect in rectangles:
                if rect.get('ContentType') != 'GraphicType':
                    continue
                slot_index += 1

                name = rect.get('Name', f'slot_{slot_index}')
                self_id = rect.get('Self', f'uFrame{slot_index}')
                transform = parse_item_transform(rect.get('ItemTransform', '1 0 0 1 0 0'))
                width_pt, height_pt = parse_path_geometry(rect, ns)

                if width_pt is None:
                    continue

                width_cm = width_pt / CM_TO_PT
                height_cm = height_pt / CM_TO_PT
                aspect_ratio = width_cm / height_cm if height_cm > 0 else 1.0

                all_slots.append(SlotInfo(
                    slot_id=self_id, name=name,
                    center_x_cm=round(transform['tx'] / CM_TO_PT, 2),
                    center_y_cm=round(transform['ty'] / CM_TO_PT, 2),
                    width_cm=round(width_cm, 2),
                    height_cm=round(height_cm, 2),
                    rotation_deg=round(transform['rotation_deg'], 2),
                    aspect_ratio=round(aspect_ratio, 3),
                    aspect_category=classify_aspect(aspect_ratio),
                    area_cm2=round(width_cm * height_cm, 2)
                ))

        return TemplateInfo(
            page_width_cm=round(page_width_pt / CM_TO_PT, 2),
            page_height_cm=round(page_height_pt / CM_TO_PT, 2),
            num_slots=len(all_slots),
            slots=all_slots
        )


def print_template_info(info: TemplateInfo):
    """הדפסה קריאה"""
    print("=" * 70)
    print(f"📐 מידע על התבנית")
    print("=" * 70)
    print(f"גודל דף: {info.page_width_cm} × {info.page_height_cm} ס\"מ")
    print(f"מספר slots: {info.num_slots}\n")

    for i, slot in enumerate(info.slots, 1):
        pos = f"({slot.center_x_cm:.1f}, {slot.center_y_cm:.1f})"
        size = f"{slot.width_cm:.1f}×{slot.height_cm:.1f}"
        rot = f"{slot.rotation_deg:+.1f}°"
        print(f"  {i}. {slot.name}: {pos}, {size}cm, {rot}, {slot.aspect_category}")

    total_area = sum(s.area_cm2 for s in info.slots)
    page_area = info.page_width_cm * info.page_height_cm
    coverage = (total_area / page_area) * 100
    print(f"\n📊 כיסוי: {coverage:.1f}% תמונות, {100-coverage:.1f}% שטח מעוצב")

print("✓ IDML Parser טעון")

✓ IDML Parser טעון


<!-- Cell 06 (Markdown) -->
<div dir=RTL>

## 🎨 שלב 2: IDML Composer

יוצר קבצי IDML חדשים - גם תבניות ריקות וגם כאלה עם תמונות משובצות.

In [4]:
# === Cell 07 ===
def create_sample_template_idml(output_path: str) -> str:
    """יוצר תבנית IDML לדוגמה - 60x30 ס"מ עם 6 slots בזוויות"""

    PAGE_WIDTH = 60 * CM_TO_PT
    PAGE_HEIGHT = 30 * CM_TO_PT

    SLOTS_DESIGN = [
        {"id": "slot_1", "cx": 340, "cy": 260, "w": 340, "h": 240, "rotation": -6},
        {"id": "slot_2", "cx": 850, "cy": 220, "w": 300, "h": 380, "rotation": 3},
        {"id": "slot_3", "cx": 1360, "cy": 260, "w": 340, "h": 240, "rotation": -4},
        {"id": "slot_4", "cx": 340, "cy": 620, "w": 300, "h": 240, "rotation": 5},
        {"id": "slot_5", "cx": 850, "cy": 650, "w": 340, "h": 260, "rotation": -3},
        {"id": "slot_6", "cx": 1360, "cy": 620, "w": 260, "h": 340, "rotation": 7},
    ]

    files_to_add = {}

    files_to_add["mimetype"] = "application/vnd.adobe.indesign-idml-package"

    files_to_add["META-INF/container.xml"] = """<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<container xmlns="urn:oasis:names:tc:opendocument:xmlns:container" version="1.0">
   <rootfiles>
      <rootfile full-path="designmap.xml" media-type="text/xml"/>
   </rootfiles>
</container>"""

    stories = " ".join([f"uStory{i}" for i in range(1, 7)])
    story_refs = "".join([f'<idPkg:Story src="Stories/Story_uStory{i}.xml"/>' for i in range(1, 7)])

    files_to_add["designmap.xml"] = f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<?aid style="50" type="document" readerVersion="6.0" featureSet="257" product="17.0(81)" ?>
<Document xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging"
          DOMVersion="17.0" Self="dDocument" StoryList="{stories}">
    <idPkg:Graphic src="Resources/Graphic.xml"/>
    <idPkg:Fonts src="Resources/Fonts.xml"/>
    <idPkg:Styles src="Resources/Styles.xml"/>
    <idPkg:Preferences src="Resources/Preferences.xml"/>
    <idPkg:Spread src="Spreads/Spread_uSpread1.xml"/>
    {story_refs}
    <Language Self="Language/English%3a+USA" Name="English: USA"/>
</Document>"""

    files_to_add["Resources/Graphic.xml"] = """<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Graphic xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <Color Self="Color/Black" Model="Process" Space="CMYK" ColorValue="0 0 0 100" ColorOverride="Specialblack" Name="Black" ColorEditable="false" ColorRemovable="false" Visible="true" SwatchCreatorID="7937"/>
    <Color Self="Color/Paper" Model="Process" Space="CMYK" ColorValue="0 0 0 0" ColorOverride="Specialpaper" Name="Paper" ColorEditable="true" ColorRemovable="false" Visible="true" SwatchCreatorID="7937"/>
</idPkg:Graphic>"""

    files_to_add["Resources/Fonts.xml"] = """<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Fonts xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <FontFamily Self="uFontFamily1" Name="Minion Pro">
        <Font Self="uFont1" FontFamily="Minion Pro" Name="Minion Pro Regular" PostScriptName="MinionPro-Regular" Status="Installed" FontStyleName="Regular" FontType="OpenTypeCFF"/>
    </FontFamily>
</idPkg:Fonts>"""

    files_to_add["Resources/Styles.xml"] = """<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Styles xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <RootCharacterStyleGroup Self="uRootChar">
        <CharacterStyle Self="CharacterStyle/$ID/[No character style]" Imported="false" Name="$ID/[No character style]"/>
    </RootCharacterStyleGroup>
    <RootParagraphStyleGroup Self="uRootPara">
        <ParagraphStyle Self="ParagraphStyle/$ID/[No paragraph style]" Imported="false" NextStyle="ParagraphStyle/$ID/[No paragraph style]" Name="$ID/[No paragraph style]"/>
    </RootParagraphStyleGroup>
    <RootObjectStyleGroup Self="uRootObj">
        <ObjectStyle Self="ObjectStyle/$ID/[None]" Imported="false" Name="$ID/[None]"/>
    </RootObjectStyleGroup>
</idPkg:Styles>"""

    files_to_add["Resources/Preferences.xml"] = """<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Preferences xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <DocumentPreference PageHeight="850.3937007874" PageWidth="1700.78740157480" PagesPerDocument="1" FacingPages="false" DocumentBleedTopOffset="8.5039370078" DocumentBleedBottomOffset="8.5039370078" DocumentBleedInsideOrLeftOffset="8.5039370078" DocumentBleedOutsideOrRightOffset="8.5039370078" PageBinding="LeftToRight"/>
    <ViewPreference RulerOrigin="SpreadOrigin" HorizontalMeasurementUnits="Centimeters" VerticalMeasurementUnits="Centimeters"/>
</idPkg:Preferences>"""

    # בניית ה-Spread עם 6 slots מסובבים
    frames_xml = ""
    for i, slot in enumerate(SLOTS_DESIGN, 1):
        cx, cy, w, h = slot["cx"], slot["cy"], slot["w"], slot["h"]
        rotation = slot["rotation"]
        slot_id = slot["id"]

        rad = math.radians(rotation)
        cos_r, sin_r = math.cos(rad), math.sin(rad)
        hw, hh = w / 2, h / 2

        frames_xml += f"""
        <Rectangle Self="uFrame{i}" ContentType="GraphicType" ItemLayer="uLayer1" Name="{slot_id}" Visible="true"
                   GradientFillStart="0 0" GradientFillLength="0" GradientFillAngle="0"
                   GradientStrokeStart="0 0" GradientStrokeLength="0" GradientStrokeAngle="0"
                   ItemTransform="{cos_r:.6f} {sin_r:.6f} {-sin_r:.6f} {cos_r:.6f} {cx} {cy}"
                   ParentStory="uStory{i}" LocalDisplaySetting="Default"
                   AppliedObjectStyle="ObjectStyle/$ID/[None]" StoryTitle="$ID/">
            <Properties>
                <PathGeometry>
                    <GeometryPathType PathOpen="false">
                        <PathPointArray>
                            <PathPointType Anchor="{-hw} {-hh}" LeftDirection="{-hw} {-hh}" RightDirection="{-hw} {-hh}"/>
                            <PathPointType Anchor="{-hw} {hh}" LeftDirection="{-hw} {hh}" RightDirection="{-hw} {hh}"/>
                            <PathPointType Anchor="{hw} {hh}" LeftDirection="{hw} {hh}" RightDirection="{hw} {hh}"/>
                            <PathPointType Anchor="{hw} {-hh}" LeftDirection="{hw} {-hh}" RightDirection="{hw} {-hh}"/>
                        </PathPointArray>
                    </GeometryPathType>
                </PathGeometry>
            </Properties>
        </Rectangle>"""

    files_to_add["Spreads/Spread_uSpread1.xml"] = f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<?aid style="50" type="spread" DOMVersion="17.0" ?>
<idPkg:Spread xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <Spread Self="uSpread1" PageCount="1" BindingLocation="0" AllowPageShuffle="true"
            ItemTransform="1 0 0 1 0 -425.197" ShowMasterItems="true"
            PageTransitionType="None" PageTransitionDirection="NotApplicable" PageTransitionDuration="Medium">
        <FlattenerPreference LineArtAndTextResolution="300" GradientAndMeshResolution="150" ClipComplexRegions="false" ConvertAllStrokesToOutlines="false" ConvertAllTextToOutlines="false">
            <Properties>
                <RasterVectorBalance type="double">50</RasterVectorBalance>
            </Properties>
        </FlattenerPreference>
        <Page Self="uPage1" Name="1" AppliedTrapPreset="TrapPreset/$ID/kDefaultTrapStyleName"
              OverrideList="" TabOrder="" GridStartingPoint="TopOutside" UseMasterGrid="true"
              GeometricBounds="0 0 {PAGE_HEIGHT} {PAGE_WIDTH}" ItemTransform="1 0 0 1 0 0">
            <Properties>
                <Descriptor type="list"/>
                <PageColor type="enumeration">UseMasterColor</PageColor>
            </Properties>
            <MarginPreference ColumnCount="1" ColumnGutter="12" Top="0" Bottom="0" Left="0" Right="0"/>
        </Page>
        {frames_xml}
    </Spread>
</idPkg:Spread>"""

    # Stories ריקים
    for i in range(1, 7):
        files_to_add[f"Stories/Story_uStory{i}.xml"] = f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<?aid style="50" type="story" DOMVersion="17.0" ?>
<idPkg:Story xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <Story Self="uStory{i}" AppliedTOCStyle="n" TrackChanges="false" StoryTitle="$ID/" AppliedNamedGrid="n">
        <StoryPreference OpticalMarginAlignment="false" OpticalMarginSize="12" FrameType="TextFrameType" StoryOrientation="Horizontal" StoryDirection="LeftToRightDirection"/>
        <InCopyExportOption IncludeGraphicProxies="true" IncludeAllResources="false"/>
    </Story>
</idPkg:Story>"""

    # אריזה כ-ZIP
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr(zipfile.ZipInfo("mimetype"), files_to_add["mimetype"], compress_type=zipfile.ZIP_STORED)
        for filename, content in files_to_add.items():
            if filename == "mimetype":
                continue
            zf.writestr(filename, content)

    return output_path


@dataclass
class ImagePlacement:
    """הנחיה למיקום תמונה ב-slot"""
    image_path: str
    slot: SlotInfo
    crop_strategy: str = "smart"


def prepare_image_for_print(source_path, output_path, target_width_cm, target_height_cm, target_dpi=300, crop_strategy="smart"):
    """הכנת תמונה להדפסה - קרופ + resize ל-DPI נכון"""
    target_width_px = int(target_width_cm / 2.54 * target_dpi)
    target_height_px = int(target_height_cm / 2.54 * target_dpi)

    img = Image.open(source_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')

    original_width, original_height = img.size
    target_ratio = target_width_px / target_height_px
    source_ratio = original_width / original_height

    if source_ratio > target_ratio:
        new_width = int(original_height * target_ratio)
        left = (original_width - new_width) // 2
        img = img.crop((left, 0, left + new_width, original_height))
    else:
        new_height = int(original_width / target_ratio)
        top = (original_height - new_height) // 2
        if crop_strategy == "smart":
            top = int(top * 0.5)
        img = img.crop((0, top, original_width, top + new_height))

    img = img.resize((target_width_px, target_height_px), Image.LANCZOS)
    img.save(output_path, 'JPEG', quality=92, dpi=(target_dpi, target_dpi))

    return {'width_px': target_width_px, 'height_px': target_height_px, 'dpi': target_dpi}


def build_idml_with_images(template: TemplateInfo, placements: List[ImagePlacement], output_path: str) -> str:
    """בונה IDML סופי עם תמונות מוטמעות"""

    temp_dir = Path(f"/tmp/idml_{uuid.uuid4().hex[:8]}")
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        for subdir in ["META-INF", "Resources", "Spreads", "Stories", "Links"]:
            (temp_dir / subdir).mkdir()

        # הכנת תמונות
        prepared_images = []
        for i, placement in enumerate(placements, 1):
            original_name = Path(placement.image_path).name
            prepared_name = f"img_{i:03d}_{original_name}"
            prepared_path = temp_dir / "Links" / prepared_name

            info = prepare_image_for_print(
                placement.image_path, str(prepared_path),
                placement.slot.width_cm, placement.slot.height_cm,
                300, placement.crop_strategy
            )

            prepared_images.append({
                'placement': placement, 'prepared_name': prepared_name, 'info': info,
                'uid_image': f"uImg{uuid.uuid4().hex[:12]}",
                'uid_rectangle': f"uRect{uuid.uuid4().hex[:12]}",
                'uid_story': f"uStory{uuid.uuid4().hex[:12]}",
            })

        # mimetype
        (temp_dir / "mimetype").write_text("application/vnd.adobe.indesign-idml-package")

        # container
        (temp_dir / "META-INF" / "container.xml").write_text("""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<container xmlns="urn:oasis:names:tc:opendocument:xmlns:container" version="1.0">
   <rootfiles>
      <rootfile full-path="designmap.xml" media-type="text/xml"/>
   </rootfiles>
</container>""")

        # designmap
        story_list = " ".join(p['uid_story'] for p in prepared_images)
        story_refs = "".join(f'<idPkg:Story src="Stories/Story_{p["uid_story"]}.xml"/>' for p in prepared_images)
        (temp_dir / "designmap.xml").write_text(f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<?aid style="50" type="document" readerVersion="6.0" featureSet="257" product="17.0(81)" ?>
<Document xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0" Self="dDocument" StoryList="{story_list}">
    <idPkg:Graphic src="Resources/Graphic.xml"/>
    <idPkg:Fonts src="Resources/Fonts.xml"/>
    <idPkg:Styles src="Resources/Styles.xml"/>
    <idPkg:Preferences src="Resources/Preferences.xml"/>
    <idPkg:Spread src="Spreads/Spread_uSpread1.xml"/>
    {story_refs}
    <Language Self="Language/English%3a+USA" Name="English: USA"/>
</Document>""")

        # Resources
        (temp_dir / "Resources" / "Graphic.xml").write_text("""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Graphic xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <Color Self="Color/Black" Model="Process" Space="CMYK" ColorValue="0 0 0 100" ColorOverride="Specialblack" Name="Black" ColorEditable="false" ColorRemovable="false" Visible="true" SwatchCreatorID="7937"/>
    <Color Self="Color/Paper" Model="Process" Space="CMYK" ColorValue="0 0 0 0" ColorOverride="Specialpaper" Name="Paper" ColorEditable="true" ColorRemovable="false" Visible="true" SwatchCreatorID="7937"/>
</idPkg:Graphic>""")

        (temp_dir / "Resources" / "Fonts.xml").write_text("""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Fonts xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <FontFamily Self="uFontFamily1" Name="Minion Pro">
        <Font Self="uFont1" FontFamily="Minion Pro" Name="Minion Pro Regular" PostScriptName="MinionPro-Regular" Status="Installed" FontStyleName="Regular" FontType="OpenTypeCFF"/>
    </FontFamily>
</idPkg:Fonts>""")

        (temp_dir / "Resources" / "Styles.xml").write_text("""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Styles xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <RootCharacterStyleGroup Self="uRootChar"><CharacterStyle Self="CharacterStyle/$ID/[No character style]" Imported="false" Name="$ID/[No character style]"/></RootCharacterStyleGroup>
    <RootParagraphStyleGroup Self="uRootPara"><ParagraphStyle Self="ParagraphStyle/$ID/[No paragraph style]" Imported="false" NextStyle="ParagraphStyle/$ID/[No paragraph style]" Name="$ID/[No paragraph style]"/></RootParagraphStyleGroup>
    <RootObjectStyleGroup Self="uRootObj"><ObjectStyle Self="ObjectStyle/$ID/[None]" Imported="false" Name="$ID/[None]"/></RootObjectStyleGroup>
</idPkg:Styles>""")

        pw_pt = template.page_width_cm * CM_TO_PT
        ph_pt = template.page_height_cm * CM_TO_PT
        (temp_dir / "Resources" / "Preferences.xml").write_text(f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<idPkg:Preferences xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <DocumentPreference PageHeight="{ph_pt:.4f}" PageWidth="{pw_pt:.4f}" PagesPerDocument="1" FacingPages="false" DocumentBleedTopOffset="8.5039370078" DocumentBleedBottomOffset="8.5039370078" DocumentBleedInsideOrLeftOffset="8.5039370078" DocumentBleedOutsideOrRightOffset="8.5039370078" PageBinding="LeftToRight"/>
    <ViewPreference RulerOrigin="SpreadOrigin" HorizontalMeasurementUnits="Centimeters" VerticalMeasurementUnits="Centimeters"/>
</idPkg:Preferences>""")

        # Spread עם תמונות
        frames_xml = ""
        for pimg in prepared_images:
            slot = pimg['placement'].slot
            cx_pt = slot.center_x_cm * CM_TO_PT
            cy_pt = slot.center_y_cm * CM_TO_PT
            w_pt = slot.width_cm * CM_TO_PT
            h_pt = slot.height_cm * CM_TO_PT

            rad = math.radians(slot.rotation_deg)
            cos_r, sin_r = math.cos(rad), math.sin(rad)
            hw, hh = w_pt / 2, h_pt / 2

            image_uri = f"file:../Links/{pimg['prepared_name']}"

            frames_xml += f"""
        <Rectangle Self="{pimg['uid_rectangle']}" ContentType="GraphicType" ItemLayer="uLayer1" Name="{slot.name}" Visible="true"
                   GradientFillStart="0 0" GradientFillLength="0" GradientFillAngle="0"
                   GradientStrokeStart="0 0" GradientStrokeLength="0" GradientStrokeAngle="0"
                   ItemTransform="{cos_r:.6f} {sin_r:.6f} {-sin_r:.6f} {cos_r:.6f} {cx_pt:.4f} {cy_pt:.4f}"
                   ParentStory="{pimg['uid_story']}" LocalDisplaySetting="Default"
                   AppliedObjectStyle="ObjectStyle/$ID/[None]" StoryTitle="$ID/">
            <Properties>
                <PathGeometry>
                    <GeometryPathType PathOpen="false">
                        <PathPointArray>
                            <PathPointType Anchor="{-hw:.4f} {-hh:.4f}" LeftDirection="{-hw:.4f} {-hh:.4f}" RightDirection="{-hw:.4f} {-hh:.4f}"/>
                            <PathPointType Anchor="{-hw:.4f} {hh:.4f}" LeftDirection="{-hw:.4f} {hh:.4f}" RightDirection="{-hw:.4f} {hh:.4f}"/>
                            <PathPointType Anchor="{hw:.4f} {hh:.4f}" LeftDirection="{hw:.4f} {hh:.4f}" RightDirection="{hw:.4f} {hh:.4f}"/>
                            <PathPointType Anchor="{hw:.4f} {-hh:.4f}" LeftDirection="{hw:.4f} {-hh:.4f}" RightDirection="{hw:.4f} {-hh:.4f}"/>
                        </PathPointArray>
                    </GeometryPathType>
                </PathGeometry>
            </Properties>
            <Image Self="{pimg['uid_image']}" ImageTypeName="$ID/JPEG"
                   ItemTransform="1 0 0 1 {-hw:.4f} {-hh:.4f}"
                   Visible="true" Name="{pimg['prepared_name']}"
                   ImageRenderingIntent="UseColorSettings" LocalDisplaySetting="Default"
                   AppliedObjectStyle="ObjectStyle/$ID/[None]">
                <Properties>
                    <GraphicBounds Left="0" Top="0" Right="{w_pt:.4f}" Bottom="{h_pt:.4f}"/>
                </Properties>
                <Link Self="uLink{pimg['uid_image']}" LinkResourceURI="{image_uri}"
                      LinkResourceFormat="$ID/JPEG" StoredState="Normal"
                      LinkImportPolicy="NoAutoImport" LinkExportPolicy="NoAutoExport"
                      AssetURL="$ID/" AssetID="$ID/"/>
            </Image>
        </Rectangle>"""

        (temp_dir / "Spreads" / "Spread_uSpread1.xml").write_text(f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<?aid style="50" type="spread" DOMVersion="17.0" ?>
<idPkg:Spread xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <Spread Self="uSpread1" PageCount="1" BindingLocation="0" AllowPageShuffle="true"
            ItemTransform="1 0 0 1 0 -425.197" ShowMasterItems="true"
            PageTransitionType="None" PageTransitionDirection="NotApplicable" PageTransitionDuration="Medium">
        <FlattenerPreference LineArtAndTextResolution="300" GradientAndMeshResolution="150" ClipComplexRegions="false" ConvertAllStrokesToOutlines="false" ConvertAllTextToOutlines="false">
            <Properties><RasterVectorBalance type="double">50</RasterVectorBalance></Properties>
        </FlattenerPreference>
        <Page Self="uPage1" Name="1" AppliedTrapPreset="TrapPreset/$ID/kDefaultTrapStyleName"
              OverrideList="" TabOrder="" GridStartingPoint="TopOutside" UseMasterGrid="true"
              GeometricBounds="0 0 {ph_pt:.4f} {pw_pt:.4f}" ItemTransform="1 0 0 1 0 0">
            <Properties><Descriptor type="list"/><PageColor type="enumeration">UseMasterColor</PageColor></Properties>
            <MarginPreference ColumnCount="1" ColumnGutter="12" Top="0" Bottom="0" Left="0" Right="0"/>
        </Page>
        {frames_xml}
    </Spread>
</idPkg:Spread>""")

        # Stories
        for pimg in prepared_images:
            (temp_dir / "Stories" / f"Story_{pimg['uid_story']}.xml").write_text(f"""<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<?aid style="50" type="story" DOMVersion="17.0" ?>
<idPkg:Story xmlns:idPkg="http://ns.adobe.com/AdobeInDesign/idml/1.0/packaging" DOMVersion="17.0">
    <Story Self="{pimg['uid_story']}" AppliedTOCStyle="n" TrackChanges="false" StoryTitle="$ID/" AppliedNamedGrid="n">
        <StoryPreference OpticalMarginAlignment="false" OpticalMarginSize="12" FrameType="TextFrameType" StoryOrientation="Horizontal" StoryDirection="LeftToRightDirection"/>
        <InCopyExportOption IncludeGraphicProxies="true" IncludeAllResources="false"/>
    </Story>
</idPkg:Story>""")

        # אריזה
        with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.write(temp_dir / "mimetype", "mimetype", compress_type=zipfile.ZIP_STORED)
            for root, dirs, files in os.walk(temp_dir):
                for file in files:
                    if file == "mimetype":
                        continue
                    file_path = Path(root) / file
                    arcname = file_path.relative_to(temp_dir)
                    zf.write(file_path, arcname)

        return output_path

    finally:
        if temp_dir.exists():
            shutil.rmtree(temp_dir)


print("✓ IDML Composer טעון")

✓ IDML Composer טעון


<!-- Cell 08 (Markdown) -->
<div dir=RTL>

## 🖼️ שלב 3: עיבוד תמונות

קוד לניתוח טכני של כל תמונה - זיהוי פנים, ציוני איכות, EXIF.

In [5]:
# === Cell 09 ===
# Face detection
FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
EYE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')


@dataclass
class FaceInfo:
    x: int
    y: int
    width: int
    height: int
    eyes_detected: int
    relative_size: float


@dataclass
class QualityScores:
    sharpness: float
    brightness: float
    contrast: float
    overall: float


@dataclass
class ImageAnalysis:
    filepath: str
    filename: str
    width: int
    height: int
    aspect_ratio: float
    aspect_category: str
    num_faces: int
    faces: List[FaceInfo]
    largest_face_size: float
    quality: QualityScores
    scene_type: str
    people_tags: List[str]
    timestamp: str
    album_score: float = 0.0


def calculate_sharpness(img_gray):
    """חדות באמצעות Laplacian variance"""
    return round(min(100, (cv2.Laplacian(img_gray, cv2.CV_64F).var() / 15)), 2)


def calculate_brightness(img):
    """בהירות ממוצעת - 50 = מאוזן"""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    brightness_norm = (np.mean(lab[:, :, 0]) / 255) * 100
    return round(max(0, 100 - abs(brightness_norm - 50) * 2), 2)


def calculate_contrast(img_gray):
    return round(min(100, (np.std(img_gray) / 70) * 100), 2)


def detect_faces(img_gray, img_shape):
    h, w = img_shape
    img_area = w * h
    faces = FACE_CASCADE.detectMultiScale(img_gray, 1.1, 5, minSize=(30, 30))

    face_infos = []
    for (fx, fy, fw, fh) in faces:
        roi = img_gray[fy:fy+fh, fx:fx+fw]
        eyes = EYE_CASCADE.detectMultiScale(roi, 1.1, 3)
        face_infos.append(FaceInfo(
            x=int(fx), y=int(fy), width=int(fw), height=int(fh),
            eyes_detected=len(eyes),
            relative_size=round((fw * fh) / img_area, 4)
        ))
    return face_infos


def analyze_image(filepath, metadata=None):
    """ניתוח מלא של תמונה בודדת"""
    img_bgr = cv2.imread(filepath)
    if img_bgr is None:
        raise ValueError(f"Cannot read: {filepath}")

    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = img_bgr.shape[:2]

    sharpness = calculate_sharpness(img_gray)
    brightness = calculate_brightness(img_bgr)
    contrast = calculate_contrast(img_gray)
    overall = round((sharpness * 0.5 + brightness * 0.3 + contrast * 0.2), 2)

    faces = detect_faces(img_gray, (h, w))
    largest_face = max([f.relative_size for f in faces], default=0.0)

    meta = metadata or {}

    return ImageAnalysis(
        filepath=filepath,
        filename=Path(filepath).name,
        width=w, height=h,
        aspect_ratio=round(w/h, 3),
        aspect_category=classify_aspect(w/h),
        num_faces=len(faces),
        faces=faces,
        largest_face_size=round(largest_face, 4),
        quality=QualityScores(sharpness, brightness, contrast, overall),
        scene_type=meta.get('scene_type', 'unknown'),
        people_tags=meta.get('people', []),
        timestamp=meta.get('timestamp', ''),
    )

print("✓ מודול עיבוד תמונות טעון")

✓ מודול עיבוד תמונות טעון


<!-- Cell 10 (Markdown) -->
<div dir=RTL>

## 🧠 שלב 4: זיהוי אנשים וסצנות

מודול שבונה על הניתוח הטכני ומוסיף **משמעות סמנטית** - מי בתמונה, באיזו סצנה.

In [6]:
# === Cell 11 ===
@dataclass
class PersonIdentity:
    person_id: str
    role: str
    importance: int  # 1-10
    face_embedding: List[float]


@dataclass
class SceneAnalysis:
    primary_scene: str
    confidence: float
    semantic_tags: List[str]
    mood: str
    indoor_outdoor: str
    composition: str


# פלטת אנשים באירוע
EVENT_PEOPLE = {
    "batmitzvah_girl": PersonIdentity("p_001", "batmitzvah_girl", 10, [0.8,0.2,0.3,0.9,0.1,0.5,0.7,0.4]),
    "mother": PersonIdentity("p_002", "mother", 8, [0.3,0.8,0.4,0.6,0.5,0.2,0.9,0.3]),
    "father": PersonIdentity("p_003", "father", 8, [0.2,0.5,0.9,0.3,0.7,0.4,0.1,0.8]),
    "grandma": PersonIdentity("p_004", "grandma", 6, [0.9,0.4,0.1,0.5,0.3,0.8,0.2,0.6]),
    "friend_1": PersonIdentity("p_005", "close_friend", 4, [0.4,0.6,0.8,0.2,0.9,0.1,0.5,0.3]),
    "friend_2": PersonIdentity("p_006", "close_friend", 4, [0.5,0.3,0.2,0.8,0.6,0.9,0.4,0.1]),
    "friend_3": PersonIdentity("p_007", "close_friend", 4, [0.6,0.9,0.5,0.1,0.4,0.3,0.8,0.2]),
    "sibling": PersonIdentity("p_008", "sibling", 7, [0.7,0.1,0.6,0.4,0.2,0.8,0.3,0.9]),
}

SCENE_SEMANTICS = {
    "preparations_makeup": {"tags": ["makeup", "mirror", "intimate"], "mood": "anticipation", "indoor_outdoor": "indoor", "composition": "detail"},
    "preparations_dress": {"tags": ["dress", "family", "helping"], "mood": "emotional", "indoor_outdoor": "indoor", "composition": "portrait"},
    "studio_portrait_batmitzvah": {"tags": ["portrait", "studio", "formal"], "mood": "formal", "indoor_outdoor": "indoor", "composition": "portrait"},
    "studio_portrait_family": {"tags": ["family", "studio", "group"], "mood": "formal", "indoor_outdoor": "indoor", "composition": "group"},
    "ceremony_arrival": {"tags": ["entrance", "venue"], "mood": "formal", "indoor_outdoor": "indoor", "composition": "wide"},
    "ceremony_speeches": {"tags": ["speech", "emotional"], "mood": "emotional", "indoor_outdoor": "indoor", "composition": "portrait"},
    "reception_guests": {"tags": ["guests", "hall"], "mood": "joyful", "indoor_outdoor": "indoor", "composition": "group"},
    "dance_floor_action": {"tags": ["dancing", "energy"], "mood": "energetic", "indoor_outdoor": "indoor", "composition": "group"},
    "dance_floor_batmitzvah": {"tags": ["dancing", "celebration"], "mood": "joyful", "indoor_outdoor": "indoor", "composition": "portrait"},
    "emotional_moment_parents": {"tags": ["parents", "hug", "tears"], "mood": "emotional", "indoor_outdoor": "indoor", "composition": "group"},
    "candid_friends": {"tags": ["friends", "laughter"], "mood": "joyful", "indoor_outdoor": "indoor", "composition": "group"},
    "detail_shot_decor": {"tags": ["decoration", "flowers"], "mood": "aesthetic", "indoor_outdoor": "indoor", "composition": "detail"},
}


def identify_faces_in_image(metadata):
    return [EVENT_PEOPLE[n] for n in metadata.get('people', []) if n in EVENT_PEOPLE]


def analyze_scene(metadata):
    scene_type = metadata.get('scene_type', 'unknown')
    s = SCENE_SEMANTICS.get(scene_type, {"tags": [], "mood": "neutral", "indoor_outdoor": "unknown", "composition": "unknown"})
    return SceneAnalysis(
        primary_scene=scene_type, confidence=0.92,
        semantic_tags=s["tags"], mood=s["mood"],
        indoor_outdoor=s["indoor_outdoor"], composition=s["composition"],
    )


def calculate_people_importance_score(people):
    if not people:
        return 0.0
    total = sum(p.importance ** 2 for p in people)
    return round(min(100, (total / 150) * 100), 2)


def has_batmitzvah_girl(people):
    return any(p.role == "batmitzvah_girl" for p in people)

print("✓ מודול זיהוי טעון")

✓ מודול זיהוי טעון


<!-- Cell 12 (Markdown) -->
<div dir=RTL>

## ⭐ שלב 5: ציוני אלבום ובחירת תמונות

איסוף כל המידע לכל תמונה + אלגוריתם בחירה.

In [7]:
# === Cell 13 ===
@dataclass
class EnrichedImage:
    filepath: str
    filename: str
    width: int
    height: int
    aspect_category: str
    num_faces_detected: int
    quality_score: float
    scene_type: str
    scene_mood: str
    scene_composition: str
    semantic_tags: List[str]
    people_roles: List[str]
    people_importance: float
    has_batmitzvah: bool
    timestamp: str
    album_score: float = 0.0
    scene_cluster: int = -1


def enrich_image(image_path, metadata):
    """העשרת תמונה עם כל המידע"""
    analysis = analyze_image(image_path, metadata)
    scene = analyze_scene(metadata)
    faces = identify_faces_in_image(metadata)

    return EnrichedImage(
        filepath=analysis.filepath, filename=analysis.filename,
        width=analysis.width, height=analysis.height,
        aspect_category=analysis.aspect_category,
        num_faces_detected=analysis.num_faces,
        quality_score=analysis.quality.overall,
        scene_type=scene.primary_scene, scene_mood=scene.mood,
        scene_composition=scene.composition, semantic_tags=scene.semantic_tags,
        people_roles=[p.role for p in faces],
        people_importance=calculate_people_importance_score(faces),
        has_batmitzvah=has_batmitzvah_girl(faces),
        timestamp=metadata.get('timestamp', ''),
    )


def calculate_album_score(img, weights=None):
    """ציון אלבום משוקלל - 0-100"""
    if weights is None:
        weights = {'quality': 0.30, 'people': 0.35, 'uniqueness': 0.20, 'emotional': 0.15}

    uniqueness_bonus = {
        "emotional_moment_parents": 90, "preparations_makeup": 75,
        "detail_shot_decor": 60, "candid_friends": 70,
        "ceremony_speeches": 80, "studio_portrait_batmitzvah": 85,
    }
    mood_scores = {
        "emotional": 90, "joyful": 80, "formal": 60,
        "aesthetic": 65, "energetic": 75, "anticipation": 70, "neutral": 40,
    }

    score = (
        img.quality_score * weights['quality'] +
        img.people_importance * weights['people'] +
        uniqueness_bonus.get(img.scene_type, 50) * weights['uniqueness'] +
        mood_scores.get(img.scene_mood, 50) * weights['emotional']
    )
    return round(score, 2)


def cluster_by_scene_and_time(images):
    """קיבוץ תמונות דומות"""
    clusters = defaultdict(list)
    sorted_imgs = sorted(images, key=lambda x: x.timestamp)

    current_cluster = 0
    last_img = None

    for img in sorted_imgs:
        if last_img is None:
            img.scene_cluster = current_cluster
            clusters[current_cluster].append(img)
            last_img = img
            continue

        same_scene = img.scene_type == last_img.scene_type
        try:
            t1 = datetime.fromisoformat(last_img.timestamp)
            t2 = datetime.fromisoformat(img.timestamp)
            time_diff_min = (t2 - t1).total_seconds() / 60
        except:
            time_diff_min = 999

        if same_scene and time_diff_min < 10:
            img.scene_cluster = current_cluster
            clusters[current_cluster].append(img)
        else:
            current_cluster += 1
            img.scene_cluster = current_cluster
            clusters[current_cluster].append(img)
        last_img = img

    return dict(clusters)


def select_images_for_album(images, target_count=96, constraints=None):
    """בחירת תמונות לאלבום"""
    if constraints is None:
        constraints = {'min_batmitzvah_pct': 0.40, 'min_scene_diversity': 8, 'max_per_cluster': 2}

    for img in images:
        img.album_score = calculate_album_score(img)

    clusters = cluster_by_scene_and_time(images)

    selected = []
    for cluster_imgs in clusters.values():
        sorted_cluster = sorted(cluster_imgs, key=lambda x: x.album_score, reverse=True)
        selected.extend(sorted_cluster[:constraints['max_per_cluster']])

    if len(selected) > target_count:
        selected = sorted(selected, key=lambda x: x.album_score, reverse=True)[:target_count]
    elif len(selected) < target_count:
        selected_names = {s.filename for s in selected}
        remaining = sorted([img for img in images if img.filename not in selected_names],
                          key=lambda x: x.album_score, reverse=True)
        selected.extend(remaining[:target_count - len(selected)])

    # איזון לאחוז בת מצווה
    batmitzvah_count = sum(1 for img in selected if img.has_batmitzvah)
    batmitzvah_pct = batmitzvah_count / len(selected) if selected else 0

    if batmitzvah_pct < constraints['min_batmitzvah_pct']:
        needed = int(len(selected) * constraints['min_batmitzvah_pct']) - batmitzvah_count
        available = sorted([img for img in images if img.has_batmitzvah
                           and img.filename not in {s.filename for s in selected}],
                          key=lambda x: x.album_score, reverse=True)
        to_remove = sorted([img for img in selected if not img.has_batmitzvah],
                          key=lambda x: x.album_score)

        for i in range(min(needed, len(available), len(to_remove))):
            selected.remove(to_remove[i])
            selected.append(available[i])

    return selected


@dataclass
class AlbumChapter:
    name: str
    scene_types: List[str]
    target_pages: int
    description: str


def organize_into_chapters(selected_images, chapters, images_per_page=6):
    """ארגון תמונות לפרקים ודפים"""
    chapter_images = {}
    used = set()

    for chapter in chapters:
        pool = sorted(
            [img for img in selected_images
             if img.scene_type in chapter.scene_types and img.filename not in used],
            key=lambda x: x.album_score, reverse=True
        )

        needed = chapter.target_pages * images_per_page
        chosen = pool[:needed]
        chosen.sort(key=lambda x: x.timestamp)

        pages = []
        for i in range(0, len(chosen), images_per_page):
            page = chosen[i:i + images_per_page]
            if page:
                pages.append(page)

        chapter_images[chapter.name] = pages
        for img in chosen:
            used.add(img.filename)

    return chapter_images

print("✓ מודול בחירה וציון טעון")

✓ מודול בחירה וציון טעון


<!-- Cell 14 (Markdown) -->
<div dir=RTL>

## 🤖 שלב 6: LLM Client

ממשק אחיד ל-Gemini (חינמי), Claude, OpenAI ו-Simulated.

In [8]:
# === Cell 15 ===
@dataclass
class LLMRequest:
    system_prompt: str
    user_prompt: str
    expected_json: bool = True
    max_tokens: int = 2000
    temperature: float = 0.3


@dataclass
class LLMResponse:
    content: str
    parsed_json: Optional[Dict] = None
    provider: str = ""
    model: str = ""
    tokens_in: int = 0
    tokens_out: int = 0
    cost_estimate_usd: float = 0.0
    error: Optional[str] = None

    @property
    def success(self):
        return self.error is None and bool(self.content)


class LLMProvider:
    name = "base"

    def complete(self, request):
        raise NotImplementedError

    def _extract_json(self, text):
        if not text:
            return None
        cleaned = text.strip()
        if cleaned.startswith('```json'):
            cleaned = cleaned[7:]
        elif cleaned.startswith('```'):
            cleaned = cleaned[3:]
        if cleaned.endswith('```'):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()

        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            start, end = cleaned.find('{'), cleaned.rfind('}')
            if start >= 0 and end > start:
                try:
                    return json.loads(cleaned[start:end+1])
                except:
                    pass
        return None


class GeminiProvider(LLMProvider):
    name = "gemini"

    def __init__(self, api_key=None, model="gemini-2.0-flash-exp"):
        self.api_key = api_key or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        self.model = model

    def complete(self, request):
        if not self.api_key:
            return LLMResponse(content="", provider=self.name, model=self.model, error="No API key")

        prompt = f"{request.system_prompt}\n\n{request.user_prompt}"
        if request.expected_json:
            prompt += "\n\nReturn ONLY valid JSON."

        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={self.api_key}"
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": request.temperature, "maxOutputTokens": request.max_tokens}
        }

        try:
            r = requests.post(url, json=payload, timeout=60)
            r.raise_for_status()
            data = r.json()
            content = data['candidates'][0]['content']['parts'][0]['text']
            usage = data.get('usageMetadata', {})

            resp = LLMResponse(
                content=content, provider=self.name, model=self.model,
                tokens_in=usage.get('promptTokenCount', 0),
                tokens_out=usage.get('candidatesTokenCount', 0),
                cost_estimate_usd=0.0
            )
            if request.expected_json:
                resp.parsed_json = self._extract_json(content)
            return resp
        except Exception as e:
            return LLMResponse(content="", provider=self.name, model=self.model, error=str(e))


class ClaudeProvider(LLMProvider):
    name = "claude"
    PRICING = {"claude-sonnet-4-6": {"input": 3.0, "output": 15.0}}

    def __init__(self, api_key=None, model="claude-sonnet-4-6"):
        self.api_key = api_key or os.environ.get("ANTHROPIC_API_KEY")
        self.model = model

    def complete(self, request):
        if not self.api_key:
            return LLMResponse(content="", provider=self.name, model=self.model, error="No API key")

        try:
            r = requests.post(
                "https://api.anthropic.com/v1/messages",
                json={
                    "model": self.model, "max_tokens": request.max_tokens,
                    "temperature": request.temperature, "system": request.system_prompt,
                    "messages": [{"role": "user", "content": request.user_prompt}]
                },
                headers={
                    "x-api-key": self.api_key,
                    "anthropic-version": "2023-06-01",
                    "content-type": "application/json",
                },
                timeout=60
            )
            r.raise_for_status()
            data = r.json()
            content = data['content'][0]['text']
            usage = data.get('usage', {})

            pricing = self.PRICING.get(self.model, {"input": 3.0, "output": 15.0})
            cost = (usage.get('input_tokens', 0) / 1e6 * pricing['input']) +                    (usage.get('output_tokens', 0) / 1e6 * pricing['output'])

            resp = LLMResponse(
                content=content, provider=self.name, model=self.model,
                tokens_in=usage.get('input_tokens', 0),
                tokens_out=usage.get('output_tokens', 0),
                cost_estimate_usd=round(cost, 6)
            )
            if request.expected_json:
                resp.parsed_json = self._extract_json(content)
            return resp
        except Exception as e:
            return LLMResponse(content="", provider=self.name, model=self.model, error=str(e))


class SimulatedProvider(LLMProvider):
    name = "simulated"

    def __init__(self, *args, **kwargs):
        self.model = "simulated-v1"

    def complete(self, request):
        content = json.dumps({
            "narrative_summary": "אלבום בת מצווה עשיר - מהכנות אינטימיות ועד חגיגה.",
            "chapters": [
                {"name": "ההכנות", "pages": 2, "scene_types": ["preparations_makeup", "preparations_dress"], "theme": "הכנות", "priority_level": 4},
                {"name": "סטודיו", "pages": 3, "scene_types": ["studio_portrait_batmitzvah", "studio_portrait_family"], "theme": "פורטרטים", "priority_level": 5},
                {"name": "רגעי שיא", "pages": 2, "scene_types": ["ceremony_speeches", "emotional_moment_parents"], "theme": "רגעים מרגשים", "priority_level": 5},
                {"name": "ריקודים", "pages": 4, "scene_types": ["dance_floor_action", "dance_floor_batmitzvah"], "theme": "חגיגה", "priority_level": 5},
            ],
            "planning_notes": "מאוזן בין רגעי עצמה רגשיים לחגיגה"
        }, ensure_ascii=False)

        resp = LLMResponse(content=content, provider=self.name, model=self.model, tokens_in=100, tokens_out=300, cost_estimate_usd=0.0)
        if request.expected_json:
            resp.parsed_json = self._extract_json(content)
        return resp


class LLMClient:
    def __init__(self, preferred_provider="auto"):
        self.providers = {}

        gemini = GeminiProvider()
        if gemini.api_key:
            self.providers['gemini'] = gemini

        claude = ClaudeProvider()
        if claude.api_key:
            self.providers['claude'] = claude

        self.providers['simulated'] = SimulatedProvider()

        if preferred_provider == "auto":
            for p in ['claude', 'gemini', 'simulated']:
                if p in self.providers:
                    self.active = self.providers[p]
                    break
        else:
            self.active = self.providers.get(preferred_provider, self.providers['simulated'])

    def complete(self, request):
        return self.active.complete(request)

    def status(self):
        return {
            "active_provider": self.active.name,
            "active_model": self.active.model,
            "available_providers": list(self.providers.keys()),
        }

print("✓ LLM Client טעון")

✓ LLM Client טעון


<!-- Cell 16 (Markdown) -->
<div dir=RTL>

## 📖 שלב 7: LLM Planner

ה"במאי" של האלבום - מתכנן מבנה אופטימלי על פי תוכן האירוע.

In [9]:
# === Cell 17 ===
@dataclass
class AlbumPlan:
    total_images_available: int
    total_pages: int
    total_images_selected: int
    chapters: List[Dict]
    narrative_summary: str
    planning_notes: str


def create_event_summary(images):
    """סיכום האירוע בפורמט מובנה ל-LLM"""
    total = len(images)
    scene_counts = Counter(img.scene_type for img in images)
    mood_counts = Counter(img.scene_mood for img in images)
    all_people = []
    for img in images:
        all_people.extend(img.people_roles)

    top_moments = sorted(images, key=lambda x: x.album_score, reverse=True)[:20]
    top_by_scene = Counter(img.scene_type for img in top_moments)

    return {
        "total_photos": total,
        "scenes_distribution": dict(scene_counts),
        "mood_distribution": dict(mood_counts),
        "people_frequency": dict(Counter(all_people)),
        "avg_technical_quality": round(sum(img.quality_score for img in images) / total, 1) if total else 0,
        "top_scenes_in_best_moments": dict(top_by_scene),
    }


def generate_album_plan(images, total_pages=16, images_per_page=6, llm_provider="auto"):
    """יצירת תוכנית אלבום"""
    event_summary = create_event_summary(images)
    total_target = total_pages * images_per_page

    client = LLMClient(preferred_provider=llm_provider)

    system_prompt = """You are an expert album designer for Bat Mitzvah events.
Respond ONLY in valid JSON format.

Output schema:
{
  "narrative_summary": "2-3 sentences in Hebrew",
  "chapters": [{"name": "Hebrew name", "pages": number, "scene_types": [...], "theme": "Hebrew", "priority_level": 1-5}],
  "planning_notes": "Observations in Hebrew"
}"""

    user_prompt = f"""Plan a {total_pages}-page album (6 photos per page, {total_target} total).

Event Data:
{json.dumps(event_summary, ensure_ascii=False, indent=2)}

Requirements:
- Total pages MUST equal {total_pages}
- Bat Mitzvah girl in at least 40% of photos
- Emotional and visual variety
- Generous space for celebration/dance
"""

    request = LLMRequest(
        system_prompt=system_prompt, user_prompt=user_prompt,
        expected_json=True, max_tokens=2000, temperature=0.4,
    )

    response = client.complete(request)

    if not response.success or not response.parsed_json:
        # fallback ל-default
        print(f"⚠️  LLM failed ({response.error}), using fallback plan")
        data = {
            "narrative_summary": "אלבום בת מצווה סטנדרטי",
            "chapters": [
                {"name": "הכנות", "pages": 2, "scene_types": ["preparations_makeup", "preparations_dress"], "theme": "", "priority_level": 4},
                {"name": "סטודיו", "pages": 3, "scene_types": ["studio_portrait_batmitzvah", "studio_portrait_family"], "theme": "", "priority_level": 5},
                {"name": "רגעי שיא", "pages": 2, "scene_types": ["ceremony_speeches", "emotional_moment_parents"], "theme": "", "priority_level": 5},
                {"name": "ריקודים", "pages": 4, "scene_types": ["dance_floor_action", "dance_floor_batmitzvah"], "theme": "", "priority_level": 5},
            ],
            "planning_notes": "fallback"
        }
    else:
        data = response.parsed_json

    chapters_output = [
        {
            "name": ch.get("name", ""), "pages": int(ch.get("pages", 0)),
            "scene_types": ch.get("scene_types", []),
            "theme": ch.get("theme", ""),
            "priority_level": int(ch.get("priority_level", 3)),
            "images_needed": int(ch.get("pages", 0)) * images_per_page,
        }
        for ch in data.get("chapters", [])
    ]

    if response.success:
        print(f"✓ תוכנית נוצרה ע\"י {response.provider}/{response.model}")
        if response.cost_estimate_usd > 0:
            print(f"  עלות: ${response.cost_estimate_usd:.4f}")

    return AlbumPlan(
        total_images_available=len(images),
        total_pages=sum(ch["pages"] for ch in chapters_output),
        total_images_selected=sum(ch["images_needed"] for ch in chapters_output),
        chapters=chapters_output,
        narrative_summary=data.get("narrative_summary", ""),
        planning_notes=data.get("planning_notes", ""),
    )


def print_album_plan(plan):
    print("\n" + "=" * 70)
    print("📖 תוכנית האלבום")
    print("=" * 70)
    print(f"\n📊 תמונות זמינות: {plan.total_images_available}, דפים: {plan.total_pages}, תמונות שיובאו: {plan.total_images_selected}")
    print(f"\n📝 נרטיב: {plan.narrative_summary}")
    print(f"💡 הערות: {plan.planning_notes}")
    print(f"\n📑 פרקים:")
    for i, ch in enumerate(plan.chapters, 1):
        print(f"  {i}. {ch['name']}: {ch['pages']} דפים, {ch['images_needed']} תמונות, {'⭐' * ch['priority_level']}")

print("✓ LLM Planner טעון")

✓ LLM Planner טעון


<!-- Cell 18 (Markdown) -->
<div dir=RTL>

## 🎯 שלב 8: התאמת תמונות ל-Slots

האלגוריתם שבוחר איזו תמונה לאיזה slot - בודק את כל 720 הפרמוטציות של 6! ובוחר אופטימלית.

In [10]:
# === Cell 19 ===
@dataclass
class ImageSlotAssignment:
    slot_id: str
    slot_name: str
    image_filename: str
    image_filepath: str
    compatibility_score: float
    crop_strategy: str


def calculate_compatibility(image, slot, is_center=False):
    """ציון התאמה תמונה-slot (0-100)"""
    score = 50.0

    # 1. התאמת אספקט
    image_aspect = image.aspect_category
    slot_aspect = slot.aspect_category

    if image_aspect == slot_aspect:
        score += 30
    elif (image_aspect == "square" or slot_aspect == "square"):
        score += 10
    else:
        score -= 20

    # 2. עלות קרופ
    image_ratio = image.width / image.height
    slot_ratio = slot.width_cm / slot.height_cm
    if slot_ratio > 0 and image_ratio > 0:
        ratio_diff = abs(slot_ratio - image_ratio) / max(slot_ratio, image_ratio)
        score -= ratio_diff * 25

    # 3. תמונות טובות ב-slot מרכזי
    if is_center:
        if image.album_score >= 75:
            score += 15
        if image.has_batmitzvah:
            score += 10

    # 4. פורטרט ב-slot אנכי
    if slot_aspect == "portrait" and image.scene_composition == "portrait":
        score += 10

    # 5. מצב קיצוני
    if (image_ratio < 0.7 and slot_ratio > 1.3) or (image_ratio > 1.3 and slot_ratio < 0.7):
        score -= 30

    return max(0, min(100, score))


def find_best_assignment(images, template):
    """מציאת השיבוץ הטוב ביותר - מנסה את כל הפרמוטציות"""
    if len(images) != len(template.slots):
        raise ValueError(f"{len(images)} images != {len(template.slots)} slots")

    # ה-slot המרכזי
    page_cx = template.page_width_cm / 2
    page_cy = template.page_height_cm / 2
    slot_centrality = {}
    for slot in template.slots:
        dist = ((slot.center_x_cm - page_cx)**2 + (slot.center_y_cm - page_cy)**2) ** 0.5
        slot_centrality[slot.slot_id] = slot.area_cm2 / (dist + 1)
    center_slot_id = max(slot_centrality.items(), key=lambda x: x[1])[0]

    best_score = -float('inf')
    best_assignment = None

    for perm in permutations(range(len(images))):
        total_score = 0
        temp_assignment = []

        for slot_idx, img_idx in enumerate(perm):
            slot = template.slots[slot_idx]
            image = images[img_idx]
            is_center = slot.slot_id == center_slot_id
            compat = calculate_compatibility(image, slot, is_center)
            weighted = compat + (image.album_score * 0.1)
            total_score += weighted

            crop = "fit" if compat >= 80 else ("smart" if compat >= 60 else "fill")

            temp_assignment.append(ImageSlotAssignment(
                slot_id=slot.slot_id, slot_name=slot.name,
                image_filename=image.filename, image_filepath=image.filepath,
                compatibility_score=round(compat, 2), crop_strategy=crop
            ))

        if total_score > best_score:
            best_score = total_score
            best_assignment = temp_assignment

    avg_score = best_score / len(images) if images else 0
    return best_assignment, round(avg_score, 2)


def print_assignment(assignment, template, avg_score):
    print(f"\n🎨 שיבוץ (ציון ממוצע: {avg_score:.1f}):")
    slot_map = {s.slot_id: s for s in template.slots}
    for a in assignment:
        slot = slot_map[a.slot_id]
        icon = "🟢" if a.compatibility_score >= 80 else ("🟡" if a.compatibility_score >= 60 else "🔴")
        print(f"  {slot.name} ({slot.width_cm:.1f}×{slot.height_cm:.1f}cm, {slot.rotation_deg:+.1f}°): {a.image_filename} {icon}{a.compatibility_score:.0f}")

print("✓ Template Matcher טעון")

✓ Template Matcher טעון


<!-- Cell 20 (Markdown) -->
<div dir=RTL>

## 🔍 שלב 9: ולידציה ורינדור

בדיקת תקינות של IDML שנוצר, ורינדור ל-PNG לתצוגה מקדימה.

In [11]:
# === Cell 21 ===
def validate_idml(idml_path):
    """בדיקת תקינות IDML - מחזיר רשימת בעיות"""
    issues = []

    if not Path(idml_path).exists():
        return [("error", "File not found")]

    try:
        with zipfile.ZipFile(idml_path, 'r') as zf:
            namelist = zf.namelist()

            required = ['mimetype', 'META-INF/container.xml', 'designmap.xml',
                       'Resources/Graphic.xml', 'Resources/Fonts.xml',
                       'Resources/Styles.xml', 'Resources/Preferences.xml']
            for r in required:
                if r not in namelist:
                    issues.append(("error", f"Missing: {r}"))

            # mimetype
            try:
                mt = zf.read('mimetype').decode('utf-8').strip()
                if mt == "application/vnd.adobe.indesign-idml-package":
                    issues.append(("info", "mimetype correct"))
                else:
                    issues.append(("error", f"Wrong mimetype: {mt}"))
            except:
                pass

            # Links
            link_files = [n for n in namelist if n.startswith('Links/')]
            if link_files:
                issues.append(("info", f"{len(link_files)} linked images"))

            # Spreads
            for sf in [n for n in namelist if n.startswith('Spreads/')]:
                try:
                    content = zf.read(sf).decode('utf-8')
                    root = ET.fromstring(content)
                    rects = root.findall('.//Rectangle')
                    images = root.findall('.//Image')
                    issues.append(("info", f"{sf}: {len(rects)} rectangles, {len(images)} images"))

                    links = root.findall('.//Link')
                    for link in links:
                        uri = link.get('LinkResourceURI', '')
                        if uri.startswith('file:../Links/'):
                            lf = uri.replace('file:../', '')
                            if lf not in namelist:
                                issues.append(("error", f"Broken link: {lf}"))
                except ET.ParseError as e:
                    issues.append(("error", f"{sf}: {e}"))

    except zipfile.BadZipFile:
        issues.append(("error", "Not a valid ZIP"))

    return issues


def print_validation(issues):
    errors = [i for i in issues if i[0] == 'error']
    warnings = [i for i in issues if i[0] == 'warning']
    infos = [i for i in issues if i[0] == 'info']

    print("=" * 70)
    print("📋 בדיקת תקינות IDML")
    print("=" * 70)

    if errors:
        for _, m in errors:
            print(f"  ❌ {m}")

    for _, m in warnings:
        print(f"  ⚠️  {m}")

    for _, m in infos:
        print(f"  ℹ️  {m}")

    print()
    if errors:
        print(f"❌ נכשל - {len(errors)} שגיאות")
        return False
    print("✅ IDML תקין!")
    return True


def render_idml_to_png(idml_path, output_png_path):
    """רינדור IDML ל-PNG באיכות גבוהה"""
    CM_TO_PX = 30

    with zipfile.ZipFile(idml_path, 'r') as zf:
        prefs = ET.fromstring(zf.read('Resources/Preferences.xml').decode('utf-8'))
        doc_pref = prefs.find('.//DocumentPreference')

        page_w_cm = float(doc_pref.get('PageWidth', 0)) / CM_TO_PT
        page_h_cm = float(doc_pref.get('PageHeight', 0)) / CM_TO_PT

        canvas_w = int(page_w_cm * CM_TO_PX)
        canvas_h = int(page_h_cm * CM_TO_PX)
        canvas = Image.new('RGBA', (canvas_w, canvas_h), (255, 253, 249, 255))

        for spread_file in [n for n in zf.namelist() if n.startswith('Spreads/')]:
            spread_root = ET.fromstring(zf.read(spread_file).decode('utf-8'))

            for rect in spread_root.findall('.//Rectangle'):
                transform = rect.get('ItemTransform', '1 0 0 1 0 0').split()
                a, b, c, d, tx, ty = [float(x) for x in transform]
                rotation_deg = math.degrees(math.atan2(b, a))
                cx_cm = tx / CM_TO_PT
                cy_cm = ty / CM_TO_PT

                path_points = rect.findall('.//PathPointType')
                xs, ys = [], []
                for pp in path_points:
                    anchor = pp.get('Anchor', '').split()
                    if len(anchor) == 2:
                        xs.append(float(anchor[0]))
                        ys.append(float(anchor[1]))

                if not xs:
                    continue

                w_cm = (max(xs) - min(xs)) / CM_TO_PT
                h_cm = (max(ys) - min(ys)) / CM_TO_PT

                image_elem = rect.find('.//Image')
                if image_elem is None:
                    continue
                link = image_elem.find('.//Link')
                if link is None:
                    continue

                image_uri = link.get('LinkResourceURI', '')
                if not image_uri.startswith('file:../Links/'):
                    continue

                image_file = image_uri.replace('file:../', '')

                try:
                    img_bytes = zf.read(image_file)
                    img = Image.open(io.BytesIO(img_bytes)).convert('RGBA')
                except:
                    continue

                slot_w_px = int(w_cm * CM_TO_PX)
                slot_h_px = int(h_cm * CM_TO_PX)
                img_resized = img.resize((slot_w_px, slot_h_px), Image.LANCZOS)

                if rotation_deg != 0:
                    img_rotated = img_resized.rotate(-rotation_deg, expand=True, resample=Image.BICUBIC)
                else:
                    img_rotated = img_resized

                cx_px = int(cx_cm * CM_TO_PX)
                cy_px = int(cy_cm * CM_TO_PX)
                paste_x = cx_px - img_rotated.width // 2
                paste_y = cy_px - img_rotated.height // 2
                canvas.paste(img_rotated, (paste_x, paste_y), img_rotated)

    final = Image.new('RGB', canvas.size, (255, 253, 249))
    final.paste(canvas, (0, 0), canvas)
    final.save(output_png_path, 'PNG', dpi=(150, 150))
    return output_png_path

print("✓ מודולי ולידציה ורינדור טעונים")

✓ מודולי ולידציה ורינדור טעונים


<!-- Cell 22 (Markdown) -->
<div dir=RTL>

## 📐 שלב 10: יצירת תבנית IDML

יוצר אוטומטית תבנית לדוגמה - 60×30 ס"מ עם 6 slots בזוויות.

> **הערה:** בעתיד תוכל להחליף את התבנית בתבנית משלך מ-Drive או מקובץ מקומי.

</div>

In [ ]:
# === Cell 23 ===
# יצירת התבנית
template_path = PROJECT_DIR / "sample_template.idml"
create_sample_template_idml(str(template_path))
print(f"✓ תבנית נוצרה: {template_path}")

# טעינה והצגה
template = parse_idml(str(template_path))
print_template_info(template)

In [ ]:
# === Cell 24 ===
# ויזואליזציה של התבנית
def visualize_template(template, scale=15):
    w = int(template.page_width_cm * scale)
    h = int(template.page_height_cm * scale)
    img = Image.new('RGB', (w + 40, h + 40), '#FAF7F2')
    draw = ImageDraw.Draw(img)
    draw.rectangle([20, 20, 20+w, 20+h], fill='white', outline='#8B7355', width=2)
    
    colors = ['#E8D5C4', '#C9D5E8', '#D4E8D5', '#E8D5DC', '#E8E0C9', '#D5C9E8']
    
    for i, slot in enumerate(template.slots):
        cx = 20 + int(slot.center_x_cm * scale)
        cy = 20 + int(slot.center_y_cm * scale)
        sw = int(slot.width_cm * scale)
        sh = int(slot.height_cm * scale)
        rad = math.radians(slot.rotation_deg)
        cos_r, sin_r = math.cos(rad), math.sin(rad)
        hw, hh = sw/2, sh/2
        corners = [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
        rotated = [(x*cos_r - y*sin_r + cx, x*sin_r + y*cos_r + cy) for x, y in corners]
        draw.polygon(rotated, fill=colors[i % 6], outline='#6B5B47')
        draw.text((cx-15, cy-5), f"Slot {i+1}", fill='#3D2E1F')
    
    return img

visualize_template(template)

<!-- Cell 25 (Markdown) -->
<div dir=RTL>

## 🔐 שלב 11: חיבור ל-Google Drive

### 📌 הערה חשובה - 2 שיטות אימות

המחברת תומכת בשתי שיטות אימות:

**1. ב-Google Colab (הכי פשוט!)** ⭐
- אין צורך ב-`client_secrets.json` בכלל
- אין צורך בהגדרות Google Cloud
- רק מאשרים אימות פשוט וזהו

**2. בסביבה מקומית (Jupyter רגיל)**
- צריך `client_secrets.json` (מהגדרות Google Cloud)
- מתבצע OAuth flow מלא דרך הדפדפן

הקוד למטה מזהה אוטומטית באיזו סביבה אתה ועובד בהתאם.

### ⚠️ חשוב!
ב-Colab, האימות יבקש גישה לחשבון Google שלך - **בחר את החשבון שאיתו אתה מחובר ל-Colab**.

</div>

In [ ]:
# === Cell 26 ===
# העלאת client_secrets.json (רק אם אתה NOT ב-Colab)
# ב-Colab - דלג על תא זה, האימות פשוט יותר!

def is_running_in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if is_running_in_colab():
    print("🌐 זוהתה סביבת Colab - לא צריך להעלות client_secrets.json!")
    print("   האימות יבוצע אוטומטית בתא הבא.")
else:
    # סביבה מקומית - צריך את הקובץ
    if Path('client_secrets.json').exists():
        print("✅ נמצא client_secrets.json בתיקייה הנוכחית")
    else:
        print("⚠️  סביבה מקומית - חסר client_secrets.json")
        print("   הורד מ-Google Cloud Console ושים בתיקייה הנוכחית")


<!-- Cell 27 (Markdown) -->
<div dir=RTL>

### 🔑 אימות מול Google Drive

עכשיו נבצע את האימות בפועל.

**אם אתה ב-Colab:**
- יפתח חלון מוקפץ קטן
- לחץ על הכפתור הכחול
- תפתח חלונית שתאשר גישה לחשבון Google שלך
- בחר את החשבון הנכון
- לחץ "Allow"

**אם אתה מקומי:**
- יפתח דפדפן עם דף אישור Google
- אשר את הגישה
- ה-token יישמר אוטומטית

</div>

In [ ]:
# === Cell 28 ===
# חיבור ל-Google Drive - אוטומטי לפי סביבה
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
TOKEN_FILE = 'token.json'


def get_drive_service_colab():
    """אימות פשוט עבור Colab - הכי קל"""
    from google.colab import auth as colab_auth
    import google.auth
    
    print("🔐 מאמת מול Google...")
    print("   ↳ יפתח חלון - אשר גישה לחשבון שלך")
    
    # זה הקסם - אימות אוטומטי של Colab
    colab_auth.authenticate_user()
    
    # קבלת credentials עם הScopes הנדרשים
    creds, _ = google.auth.default(scopes=SCOPES)
    
    return build('drive', 'v3', credentials=creds)


def get_drive_service_local():
    """אימות עבור סביבה מקומית - דורש client_secrets.json"""
    from google_auth_oauthlib.flow import InstalledAppFlow
    
    CREDENTIALS_FILE = 'client_secrets.json'
    creds = None
    
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(CREDENTIALS_FILE):
                raise FileNotFoundError(
                    f"לא נמצא {CREDENTIALS_FILE}!\n"
                    "הורד מ-Google Cloud Console ושים בתיקייה הנוכחית."
                )
            
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open(TOKEN_FILE, 'w') as token:
            token.write(creds.to_json())
    
    return build('drive', 'v3', credentials=creds)


# יצירת החיבור - אוטומטי לפי סביבה
try:
    import google.colab
    print("🌐 סביבת Colab")
    drive_service = get_drive_service_colab()
except ImportError:
    print("💻 סביבה מקומית")
    drive_service = get_drive_service_local()

print("✅ חיבור ל-Google Drive פעיל")

# בדיקה - מציג את שם המשתמש המחובר
about = drive_service.about().get(fields='user').execute()
print(f"👤 מחובר כ: {about['user']['emailAddress']}")


<!-- Cell 29 (Markdown) -->
<div dir=RTL>

## 📥 שלב 12: הורדת תמונות מ-Google Drive

### איך לתת קישור לתיקייה?

1. עבור לתיקייה ב-Drive שמכילה את התמונות
2. לחץ ימני → "Get link" (או העתק מהכתובת בדפדפן)
3. הקישור נראה כך: `https://drive.google.com/drive/folders/1AbCdEfGhIjKl_MnOp123456`
4. ה-Folder ID הוא החלק האחרון: `1AbCdEfGhIjKl_MnOp123456`

הדבק את הקישור או ה-ID בלבד בתא הבא.

</div>

In [ ]:
# === Cell 30 ===
# פונקציות עזר להורדה מ-Drive
# תומך בכל סוגי התיקיות:
#   - My Drive (התיקיות שלך)
#   - Shared with me (תיקיות ששיתפו איתך)
#   - Shared Drives (דרייבים משותפים של ארגון)

def extract_folder_id(url_or_id: str) -> str:
    """חולץ Folder ID מקישור או מחזיר את ה-ID אם כבר ניתן"""
    url_or_id = url_or_id.strip()
    
    if 'folders/' in url_or_id:
        # זה URL מלא: https://drive.google.com/drive/folders/ABC123
        folder_id = url_or_id.split('folders/')[1].split('?')[0].split('/')[0]
    elif '/' in url_or_id:
        folder_id = url_or_id.split('/')[-1].split('?')[0]
    else:
        # זה כבר ID נקי
        folder_id = url_or_id
    
    return folder_id


def get_folder_info(service, folder_id: str) -> Dict:
    """מקבל פרטי תיקייה - שם, owner, וכו'"""
    try:
        return service.files().get(
            fileId=folder_id,
            fields='id, name, owners, shared, sharingUser, capabilities',
            supportsAllDrives=True
        ).execute()
    except Exception as e:
        print(f"⚠️ לא ניתן לגשת לתיקייה: {e}")
        return None


def list_images_in_folder(service, folder_id: str) -> List[Dict]:
    """
    מחזיר רשימת קובצי תמונה בתיקייה.
    תומך בכל סוגי הדרייבים (My Drive / Shared / Shared Drives).
    """
    
    # סינון: תמונות, לא נמחקו
    query = (f"'{folder_id}' in parents and "
             f"(mimeType='image/jpeg' or mimeType='image/png' or "
             f"mimeType='image/jpg' or mimeType='image/heic') and "
             f"trashed=false")
    
    all_files = []
    page_token = None
    
    while True:
        results = service.files().list(
            q=query,
            spaces='drive',
            fields='nextPageToken, files(id, name, mimeType, size, modifiedTime, imageMediaMetadata, owners)',
            pageToken=page_token,
            pageSize=100,
            # ⭐ קריטי - תמיכה בתיקיות משותפות וב-Shared Drives:
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            corpora='allDrives'  # מחפש בכל הדרייבים שיש לך גישה אליהם
        ).execute()
        
        all_files.extend(results.get('files', []))
        page_token = results.get('nextPageToken')
        
        if not page_token:
            break
    
    return all_files


def download_image(service, file_id: str, output_path: str) -> bool:
    """מוריד קובץ בודד מהדרייב"""
    try:
        request = service.files().get_media(
            fileId=file_id,
            supportsAllDrives=True  # ⭐ תמיכה בכל סוגי הדרייבים
        )
        with io.FileIO(output_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        return True
    except Exception as e:
        print(f"  ⚠️ שגיאה בהורדת {file_id}: {e}")
        return False


def download_images_from_folder(service, folder_url_or_id: str, output_dir: Path, max_images: int = None) -> List[Dict]:
    """
    מוריד את כל התמונות מתיקייה בדרייב.
    מחזיר רשימת metadata על התמונות שירדו.
    
    תומך ב:
    - תיקיות ב-My Drive שלך
    - תיקיות ששיתפו איתך (Shared with me)
    - תיקיות ב-Shared Drives של ארגון
    """
    folder_id = extract_folder_id(folder_url_or_id)
    print(f"📁 Folder ID: {folder_id}")
    
    # קודם בודקים שיש לנו גישה לתיקייה
    folder_info = get_folder_info(service, folder_id)
    if folder_info:
        owner = folder_info.get('owners', [{}])[0].get('emailAddress', 'unknown')
        is_shared = folder_info.get('shared', False)
        share_indicator = "🔗 משותף" if is_shared else "👤 שלך"
        print(f"📂 תיקייה: '{folder_info['name']}' ({share_indicator}, owner: {owner})")
    
    print(f"\n🔍 סורק תיקייה...")
    files = list_images_in_folder(service, folder_id)
    
    if not files:
        print("⚠️  לא נמצאו תמונות בתיקייה!")
        print("   בדוק:")
        print("   1. שיש תמונות (.jpg/.png) ישירות בתיקייה (לא בתת-תיקיות)")
        print("   2. שיש לך הרשאת Viewer לפחות לתיקייה")
        return []
    
    print(f"✓ נמצאו {len(files)} תמונות")
    
    # הגבלה אם נדרש
    if max_images and len(files) > max_images:
        print(f"⚠️  מוריד רק {max_images} ראשונות (מתוך {len(files)})")
        files = files[:max_images]
    
    print(f"\n📥 מוריד תמונות ל: {output_dir}\n")
    
    output_dir.mkdir(exist_ok=True)
    metadata_list = []
    
    for i, file_info in enumerate(files, 1):
        local_path = output_dir / file_info['name']
        
        # אם הקובץ כבר קיים - דילוג
        if local_path.exists() and local_path.stat().st_size > 0:
            print(f"  [{i:3d}/{len(files)}] ⊝ {file_info['name']} (קיים)")
        else:
            success = download_image(service, file_info['id'], str(local_path))
            if not success:
                continue
            print(f"  [{i:3d}/{len(files)}] ✓ {file_info['name']}")
        
        # יצירת metadata בסיסי
        try:
            with Image.open(local_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"     ⚠️ לא ניתן לקרוא: {e}")
            continue
        
        # זמן צילום מ-EXIF (אם זמין) או מהקובץ
        timestamp = file_info.get('imageMediaMetadata', {}).get('time')
        if not timestamp:
            timestamp = file_info.get('modifiedTime', datetime.now().isoformat())
        
        metadata_list.append({
            'filename': file_info['name'],
            'filepath': str(local_path),
            'drive_file_id': file_info['id'],
            'width': width,
            'height': height,
            'aspect': 'portrait' if height > width else 'landscape',
            'aspect_category': classify_aspect(width / height),
            'timestamp': timestamp,
            # שדות שיתמלאו ע"י תיוג LLM:
            'scene_type': None,
            'people': [],
        })
    
    print(f"\n✅ הורדו {len(metadata_list)} תמונות בהצלחה")
    return metadata_list

print("✓ פונקציות הורדה מוכנות (תומך בתיקיות משותפות)")


In [ ]:
# === Cell 31 ===
# === הורדת התמונות מהדרייב ===

# הזן כאן את הקישור לתיקייה או את ה-ID
DRIVE_FOLDER = input("📂 הדבק קישור לתיקיית התמונות בדרייב (או folder ID): ")

# הגבלה למספר תמונות (אופציונלי - לבדיקות עם פחות תמונות)
# שים None להוריד הכל, או מספר כמו 30 לבדיקה ראשונה
MAX_IMAGES_FOR_TEST = 30  # שנה ל-None לאירוע מלא

# הורדה
all_metadata = download_images_from_folder(
    drive_service, 
    DRIVE_FOLDER, 
    DOWNLOADED_IMAGES_DIR,
    max_images=MAX_IMAGES_FOR_TEST
)

# שמירת metadata לcache
metadata_path = DOWNLOADED_IMAGES_DIR / "drive_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2, default=str)

print(f"\n💾 metadata נשמר: {metadata_path}")

<!-- Cell 32 (Markdown) -->
<div dir=RTL>

## 🤖 שלב 13: תיוג סצנות עם LLM Vision

עכשיו ה-LLM יסתכל על כל תמונה ויחזיר:
- **סוג סצנה**: ההכנות, סטודיו, ריקודים, נאומים, וכו'
- **אנשים בתמונה**: בת המצווה, הורים, חברים, וכו' (זיהוי תפקידים, לא שמות)
- **רגש/מצב רוח**: רגשי, חגיגי, אינטימי, וכו'

### עלות משוערת

- **Gemini Flash**: ~$0.001 לתמונה → 30 תמונות = ~$0.03
- **Claude Sonnet**: ~$0.015 לתמונה → 30 תמונות = ~$0.45
- **תוצאות נשמרות במטמון** - לא תשלם פעמיים על אותה תמונה

</div>

In [ ]:
# === Cell 33 ===
# מודול תיוג סצנות עם LLM Vision
SCENE_TYPES_HEBREW = {
    'preparations_makeup': 'הכנות - איפור',
    'preparations_dress': 'הכנות - שמלה',
    'studio_portrait_batmitzvah': 'פורטרט סטודיו - בת מצווה',
    'studio_portrait_family': 'פורטרט סטודיו - משפחה',
    'studio_portrait_friends': 'פורטרט סטודיו - חברות',
    'ceremony_arrival': 'הגעה / כניסה לאולם',
    'ceremony_speeches': 'נאומים / טקס',
    'ceremony_candles': 'הדלקת נרות / טקס דתי',
    'reception_guests': 'קבלת פנים / אורחים',
    'dance_floor_action': 'רחבת ריקודים - אקשן',
    'dance_floor_batmitzvah': 'רחבת ריקודים - בת מצווה',
    'emotional_moment_parents': 'רגע רגשי עם ההורים',
    'emotional_moment_family': 'רגע רגשי משפחתי',
    'candid_friends': 'רגע ספונטני עם חברות',
    'detail_shot_decor': 'פרט / קישוט',
    'detail_shot_food': 'פרט / אוכל',
    'wide_shot_venue': 'שוט רחב של האולם',
    'other': 'אחר',
}

PEOPLE_ROLES = {
    'batmitzvah_girl': 'בת המצווה (הילדה במרכז)',
    'mother': 'אמא',
    'father': 'אבא',
    'sibling': 'אח/אחות',
    'grandparent': 'סבא/סבתא',
    'close_friend': 'חברה קרובה',
    'extended_family': 'משפחה מורחבת',
    'guest': 'אורח/ת',
    'unknown': 'לא מזוהה',
}


def encode_image_for_llm(image_path: str, max_size_kb: int = 200) -> str:
    """
    מקודד תמונה ל-base64 עבור LLM, עם דחיסה.
    מחזיר את הbase64 string.
    """
    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # שינוי גודל - LLM לא צריך תמונה ב-300dpi לזיהוי
    max_dimension = 1024
    if max(img.size) > max_dimension:
        img.thumbnail((max_dimension, max_dimension), Image.LANCZOS)
    
    # שמירה ל-buffer עם דחיסה איטרטיבית
    quality = 90
    while quality > 30:
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        size_kb = buffer.tell() / 1024
        if size_kb <= max_size_kb:
            break
        quality -= 10
    
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode('utf-8')


SCENE_TAGGING_SYSTEM_PROMPT = """You are an expert photo analyst specializing in Bat Mitzvah events.

For each image, identify:
1. SCENE TYPE - what's happening (use ONLY codes from the provided list)
2. PEOPLE PRESENT - their roles (NOT names, just roles)
3. MOOD - the emotional tone

Available scene types: preparations_makeup, preparations_dress, studio_portrait_batmitzvah, studio_portrait_family, studio_portrait_friends, ceremony_arrival, ceremony_speeches, ceremony_candles, reception_guests, dance_floor_action, dance_floor_batmitzvah, emotional_moment_parents, emotional_moment_family, candid_friends, detail_shot_decor, detail_shot_food, wide_shot_venue, other

Available people roles: batmitzvah_girl, mother, father, sibling, grandparent, close_friend, extended_family, guest, unknown

Available moods: emotional, joyful, formal, energetic, intimate, anticipation, aesthetic, neutral

Respond ONLY in valid JSON format, no other text."""


def tag_single_image_with_llm(image_path: str, llm_client) -> Dict:
    """
    שולח תמונה ל-LLM ומקבל תיוג סמנטי.
    """
    # קידוד התמונה
    image_b64 = encode_image_for_llm(image_path, SCENE_TAGGING_CONFIG['max_image_size_kb'])
    
    user_prompt = """Analyze this photo from a Bat Mitzvah event.

Return JSON with this exact structure:
{
  "scene_type": "code from list",
  "people": ["role1", "role2"],
  "mood": "code from list",
  "confidence": 0.85,
  "description": "brief Hebrew description (1 sentence)"
}

Be precise. If you're unsure about people roles, use "unknown"."""
    
    # שליחת בקשה - תלוי בספק
    if llm_client.active.name == 'gemini':
        return _tag_with_gemini_vision(image_b64, llm_client.active, user_prompt)
    elif llm_client.active.name == 'claude':
        return _tag_with_claude_vision(image_b64, llm_client.active, user_prompt)
    else:
        # fallback - simulated
        return {
            "scene_type": "other",
            "people": ["unknown"],
            "mood": "neutral",
            "confidence": 0.0,
            "description": "[simulated - no real LLM]"
        }


def _tag_with_gemini_vision(image_b64: str, gemini_provider, user_prompt: str) -> Dict:
    """תיוג עם Gemini Vision"""
    if not gemini_provider.api_key:
        return {"error": "no api key"}
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{gemini_provider.model}:generateContent?key={gemini_provider.api_key}"
    
    payload = {
        "contents": [{
            "parts": [
                {"text": SCENE_TAGGING_SYSTEM_PROMPT + "\n\n" + user_prompt},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64
                    }
                }
            ]
        }],
        "generationConfig": {
            "temperature": 0.2,
            "maxOutputTokens": 500,
        }
    }
    
    try:
        r = requests.post(url, json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()
        text = data['candidates'][0]['content']['parts'][0]['text']
        
        # חילוץ JSON
        text = text.strip()
        if text.startswith('```'):
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        text = text.strip().rstrip('`').strip()
        
        return json.loads(text)
    except Exception as e:
        return {"error": str(e)}


def _tag_with_claude_vision(image_b64: str, claude_provider, user_prompt: str) -> Dict:
    """תיוג עם Claude Vision"""
    if not claude_provider.api_key:
        return {"error": "no api key"}
    
    payload = {
        "model": claude_provider.model,
        "max_tokens": 500,
        "system": SCENE_TAGGING_SYSTEM_PROMPT,
        "messages": [{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_b64
                    }
                },
                {"type": "text", "text": user_prompt}
            ]
        }]
    }
    
    try:
        r = requests.post(
            "https://api.anthropic.com/v1/messages",
            json=payload,
            headers={
                "x-api-key": claude_provider.api_key,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json"
            },
            timeout=30
        )
        r.raise_for_status()
        data = r.json()
        text = data['content'][0]['text'].strip()
        
        if text.startswith('```'):
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        text = text.strip().rstrip('`').strip()
        
        return json.loads(text)
    except Exception as e:
        return {"error": str(e)}


def tag_all_images(metadata_list: List[Dict], cache_path: str = None) -> List[Dict]:
    """
    מתייג את כל התמונות.
    משתמש ב-cache כדי לא לתייג את אותן תמונות פעמיים.
    """
    # טעינת cache קיים
    cache = {}
    if cache_path and Path(cache_path).exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
            cache = json.load(f)
        print(f"📂 נטען cache עם {len(cache)} תיוגים קודמים")
    
    # יצירת LLM client
    llm_client = LLMClient(preferred_provider=LLM_CONFIG['provider'])
    print(f"🤖 משתמש ב: {llm_client.active.name} ({llm_client.active.model})")
    
    if llm_client.active.name == 'simulated':
        print("⚠️  אין LLM אמיתי - התיוגים יהיו ריקים")
        print("   הגדר GEMINI_API_KEY או ANTHROPIC_API_KEY")
    
    # תיוג
    tagged = []
    new_count = 0
    cached_count = 0
    
    for i, meta in enumerate(metadata_list, 1):
        filename = meta['filename']
        
        if filename in cache:
            tags = cache[filename]
            cached_count += 1
            status = "⊝"
        else:
            print(f"  [{i:3d}/{len(metadata_list)}] 🔍 {filename}...", end='', flush=True)
            try:
                tags = tag_single_image_with_llm(meta['filepath'], llm_client)
                cache[filename] = tags
                new_count += 1
                
                # שמירת cache כל 5 תמונות
                if cache_path and new_count % 5 == 0:
                    with open(cache_path, 'w', encoding='utf-8') as f:
                        json.dump(cache, f, ensure_ascii=False, indent=2)
                
                if 'error' in tags:
                    status = f"❌ {tags['error'][:50]}"
                else:
                    status = f"✓ {tags.get('scene_type', '?')}"
                print(f" {status}")
            except Exception as e:
                tags = {"error": str(e)}
                print(f" ❌ {e}")
        
        # שילוב התיוגים ל-metadata
        meta_tagged = meta.copy()
        if 'error' not in tags:
            meta_tagged['scene_type'] = tags.get('scene_type', 'other')
            meta_tagged['people'] = tags.get('people', ['unknown'])
            meta_tagged['mood'] = tags.get('mood', 'neutral')
            meta_tagged['llm_description'] = tags.get('description', '')
            meta_tagged['llm_confidence'] = tags.get('confidence', 0)
        else:
            meta_tagged['scene_type'] = 'other'
            meta_tagged['people'] = ['unknown']
            meta_tagged['mood'] = 'neutral'
            meta_tagged['tagging_error'] = tags['error']
        
        tagged.append(meta_tagged)
    
    # שמירה סופית של cache
    if cache_path:
        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ תיוג הושלם: {cached_count} מהמטמון, {new_count} חדשים")
    return tagged

print("✓ מודול תיוג מוכן")

In [ ]:
# === Cell 34 ===
# הרצת התיוג
cache_file = DOWNLOADED_IMAGES_DIR / "tagging_cache.json"
all_metadata = tag_all_images(all_metadata, str(cache_file))

# סיכום
scene_counts = Counter(m.get('scene_type', 'unknown') for m in all_metadata)
print("\n📊 התפלגות סצנות:")
for scene, count in sorted(scene_counts.items(), key=lambda x: -x[1]):
    hebrew = SCENE_TYPES_HEBREW.get(scene, scene)
    bar = '■' * count
    print(f"  {hebrew:<35} {bar} ({count})")

# שמירה למטה-דטה
metadata_path = DOWNLOADED_IMAGES_DIR / "drive_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2, default=str)
print(f"\n💾 נשמר: {metadata_path}")

<!-- Cell 35 (Markdown) -->
<div dir=RTL>

## 🧠 שלב 14: התאמת מודול הזיהוי לתמונות אמיתיות

מודול הזיהוי המקורי הוגדר עם שמות סינתטיים ("friend_1", "friend_2"...).
ה-LLM מחזיר תפקידים גנריים ("close_friend"). נעדכן את המודול להתאים.

</div>

In [ ]:
# === Cell 36 ===
# עדכון EVENT_PEOPLE לפי תפקידים שה-LLM מחזיר
# (במקום שמות ספציפיים כמו friend_1, friend_2)

EVENT_PEOPLE = {
    "batmitzvah_girl": PersonIdentity("p_001", "batmitzvah_girl", 10, [0]*8),
    "mother": PersonIdentity("p_002", "mother", 8, [0]*8),
    "father": PersonIdentity("p_003", "father", 8, [0]*8),
    "sibling": PersonIdentity("p_004", "sibling", 7, [0]*8),
    "grandparent": PersonIdentity("p_005", "grandparent", 6, [0]*8),
    "close_friend": PersonIdentity("p_006", "close_friend", 4, [0]*8),
    "extended_family": PersonIdentity("p_007", "extended_family", 5, [0]*8),
    "guest": PersonIdentity("p_008", "guest", 2, [0]*8),
    "unknown": PersonIdentity("p_009", "unknown", 1, [0]*8),
}

# גם נעדכן את ה-SCENE_SEMANTICS עם הסצנות החדשות
SCENE_SEMANTICS.update({
    "studio_portrait_friends": {"tags": ["friends", "studio"], "mood": "joyful", "indoor_outdoor": "indoor", "composition": "group"},
    "ceremony_candles": {"tags": ["candles", "ceremony", "ritual"], "mood": "emotional", "indoor_outdoor": "indoor", "composition": "portrait"},
    "emotional_moment_family": {"tags": ["family", "emotional"], "mood": "emotional", "indoor_outdoor": "indoor", "composition": "group"},
    "detail_shot_food": {"tags": ["food", "table"], "mood": "aesthetic", "indoor_outdoor": "indoor", "composition": "detail"},
    "wide_shot_venue": {"tags": ["venue", "wide", "atmosphere"], "mood": "aesthetic", "indoor_outdoor": "indoor", "composition": "wide"},
    "other": {"tags": [], "mood": "neutral", "indoor_outdoor": "unknown", "composition": "unknown"},
})

print(f"✓ EVENT_PEOPLE עודכן ({len(EVENT_PEOPLE)} תפקידים)")
print(f"✓ SCENE_SEMANTICS עודכן ({len(SCENE_SEMANTICS)} סצנות)")

<!-- Cell 37 (Markdown) -->
<div dir=RTL>

## 🚀 שלב 15: הצינור המלא

עכשיו שיש לנו תמונות אמיתיות מתויגות, נריץ את כל הצינור.

</div>

In [ ]:
# === Cell 38 ===
# עיבוד כל התמונות
print(f"🔄 מעבד {len(all_metadata)} תמונות...\n")
start = time.time()

enriched = []
for meta in all_metadata:
    img = enrich_image(meta['filepath'], meta)
    img.album_score = calculate_album_score(img)
    enriched.append(img)

elapsed = time.time() - start
print(f"✅ הושלם ב-{elapsed:.2f} שניות ({elapsed/len(enriched)*1000:.0f}ms לתמונה)")

In [ ]:
# === Cell 39 ===
# 10 התמונות המובילות
sorted_imgs = sorted(enriched, key=lambda x: x.album_score, reverse=True)

print("🌟 10 התמונות המובילות:\n")
for i, img in enumerate(sorted_imgs[:10], 1):
    bm = "✓" if img.has_batmitzvah else "-"
    print(f"  {i:2}. {img.filename} | ציון: {img.album_score:5.1f} | {img.scene_type:<27} | בת מצווה: {bm}")

In [ ]:
# === Cell 40 ===
# היסטוגרמה של ציונים
scores = [img.album_score for img in enriched]
print("📊 התפלגות ציונים:\n")
for low, high in [(0,20), (20,40), (40,60), (60,80), (80,100)]:
    count = sum(1 for s in scores if low <= s < high)
    bar = '█' * count
    print(f"  {low:3}-{high:3}: {bar} ({count})")

print(f"\n  ממוצע: {sum(scores)/len(scores):.1f}  |  מקס: {max(scores):.1f}  |  מין: {min(scores):.1f}")

In [ ]:
# === Cell 41 ===
# LLM Client - סטטוס
client = LLMClient(preferred_provider=LLM_CONFIG['provider'])
status = client.status()

print("🤖 סטטוס LLM:")
print(f"  פעיל: {status['active_provider']} ({status['active_model']})")
print(f"  זמינים: {', '.join(status['available_providers'])}")

if status['active_provider'] == 'simulated':
    print("\n💡 להפעלת Gemini חינמי:")
    print("   1. https://aistudio.google.com/apikey")
    print("   2. צור API key")
    print("   3. os.environ['GEMINI_API_KEY'] = 'YOUR_KEY'")
    print("   4. הרץ מחדש")

In [ ]:
# === Cell 42 ===
# תכנון האלבום
demo_pages = min(ALBUM_CONFIG['total_pages'], len(enriched) // ALBUM_CONFIG['images_per_page'])
print(f"📖 מתכנן אלבום של {demo_pages} דפים...\n")

plan = generate_album_plan(enriched, total_pages=demo_pages, images_per_page=6, llm_provider=LLM_CONFIG['provider'])
print_album_plan(plan)

In [ ]:
# === Cell 43 ===
# בחירת תמונות
selected = select_images_for_album(
    enriched,
    target_count=plan.total_images_selected,
    constraints={'min_batmitzvah_pct': 0.40, 'min_scene_diversity': 6, 'max_per_cluster': 2}
)

print(f"\n✅ נבחרו {len(selected)} תמונות")

# סטטיסטיקות
bm = sum(1 for img in selected if img.has_batmitzvah)
scenes_used = Counter(img.scene_type for img in selected)

print(f"\n📈 סטטיסטיקות:")
print(f"  אחוז בת מצווה: {bm/len(selected):.0%}")
print(f"  מגוון סצנות: {len(scenes_used)}")
print(f"  ציון ממוצע: {sum(img.album_score for img in selected)/len(selected):.1f}")

print(f"\n🎬 התפלגות:")
for scene, count in sorted(scenes_used.items(), key=lambda x: -x[1]):
    print(f"  {scene}: {'■' * count} ({count})")

In [ ]:
# === Cell 44 ===
# ארגון לפרקים
chapters = [AlbumChapter(ch['name'], ch['scene_types'], ch['pages'], ch.get('theme', ''))
            for ch in plan.chapters]

album_structure = organize_into_chapters(selected, chapters, images_per_page=6)

print("📑 מבנה האלבום:\n")
for chapter_name, pages in album_structure.items():
    if not pages:
        continue
    total = sum(len(p) for p in pages)
    print(f"📖 {chapter_name}: {len(pages)} דפים, {total} תמונות")
    for i, page in enumerate(pages, 1):
        status = "✓" if len(page) == 6 else "⚠️"
        print(f"  {status} דף {i}: {len(page)} תמונות")

In [ ]:
# === Cell 45 ===
# התאמת תמונות ל-slots - הדף הראשון המלא
full_pages = []
for chapter_name, pages in album_structure.items():
    for page in pages:
        if len(page) == 6:
            full_pages.append((chapter_name, page))

if not full_pages:
    # אם אין דף מלא, ניקח 6 תמונות הכי טובות
    print("⚠️  אין דף עם 6 תמונות מלאות, לוקחים את 6 הטובות ביותר")
    top_6 = sorted(enriched, key=lambda x: x.album_score, reverse=True)[:6]
    full_pages = [("Demo", top_6)]

chapter_name, demo_page = full_pages[0]
print(f"🎯 שיבוץ לדף של פרק '{chapter_name}'\n")

for img in demo_page:
    print(f"  • {img.filename} ({img.aspect_category}, ציון {img.album_score:.1f})")

assignment, avg_score = find_best_assignment(demo_page, template)
print_assignment(assignment, template, avg_score)

In [ ]:
# === Cell 46 ===
# יצירת IDML סופי
slot_map = {s.slot_id: s for s in template.slots}
placements = [
    ImagePlacement(
        image_path=a.image_filepath,
        slot=slot_map[a.slot_id],
        crop_strategy=a.crop_strategy
    )
    for a in assignment
]

output_idml = OUTPUT_DIR / "my_album_page.idml"
print(f"🔨 בונה IDML...")
build_idml_with_images(template, placements, str(output_idml))

size_kb = os.path.getsize(output_idml) / 1024
print(f"\n🎉 IDML נוצר!")
print(f"  נתיב: {output_idml}")
print(f"  גודל: {size_kb:.1f} KB")

In [ ]:
# === Cell 47 ===
# ולידציה
issues = validate_idml(str(output_idml))
print_validation(issues)

In [ ]:
# === Cell 48 ===
# תצוגה מקדימה
png_output = OUTPUT_DIR / "album_preview.png"
render_idml_to_png(str(output_idml), str(png_output))

print(f"✓ תצוגה נשמרה: {png_output}")

# הצגה
Image.open(png_output)

<!-- Cell 49 (Markdown) -->
<div dir=RTL>

## 📤 שלב 16: העלאת האלבום הסופי לדרייב

נעלה את ה-IDML והתצוגה המקדימה חזרה לדרייב, לתיקייה חדשה.

</div>

In [ ]:
# === Cell 50 ===
def create_drive_folder(service, folder_name: str, parent_id: str = None) -> str:
    """יוצר תיקייה חדשה בדרייב, מחזיר את ה-ID שלה"""
    file_metadata = {
        'name': folder_name,
        'mimeType': 'application/vnd.google-apps.folder',
    }
    if parent_id:
        file_metadata['parents'] = [parent_id]
    
    folder = service.files().create(body=file_metadata, fields='id').execute()
    return folder.get('id')


def upload_file_to_drive(service, local_path: str, drive_folder_id: str, drive_filename: str = None) -> str:
    """מעלה קובץ לדרייב, מחזיר את ה-ID של הקובץ ואת קישור השיתוף"""
    drive_filename = drive_filename or Path(local_path).name
    
    file_metadata = {
        'name': drive_filename,
        'parents': [drive_folder_id],
    }
    
    media = MediaFileUpload(local_path, resumable=True)
    
    file = service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id, webViewLink'
    ).execute()
    
    return file.get('id'), file.get('webViewLink')


# === העלאת התוצאות ===

# הגדרת שם תיקיית הפלט בדרייב
album_folder_name = f"album_output_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# היכן ליצור? אופציה 1: כמו תיקיית התמונות (parent_id מהקובץ הראשון)
# אופציה 2: בשורש המייל
parent_folder_id = extract_folder_id(DRIVE_FOLDER)

# יצירת תיקיית פלט
print(f"📁 יוצר תיקיית פלט בדרייב: {album_folder_name}")
output_folder_id = create_drive_folder(drive_service, album_folder_name, parent_folder_id)
print(f"✓ תיקייה נוצרה: ID={output_folder_id}")

# העלאת ה-IDML הסופי
print(f"\n📤 מעלה את ה-IDML...")
idml_id, idml_link = upload_file_to_drive(drive_service, str(output_idml), output_folder_id)
print(f"✓ הועלה: {idml_link}")

# העלאת התצוגה
print(f"\n📤 מעלה תצוגה מקדימה...")
png_id, png_link = upload_file_to_drive(drive_service, str(png_output), output_folder_id)
print(f"✓ הועלה: {png_link}")

# סיכום
print(f"\n🎉 הכל הועלה לדרייב!")
print(f"\n📁 תיקיית פלט: https://drive.google.com/drive/folders/{output_folder_id}")
print(f"\n📄 IDML: {idml_link}")
print(f"🖼️  תצוגה: {png_link}")

<!-- Cell 51 (Markdown) -->
<div dir=RTL>

## 🎊 סיכום

### מה הצלחת לעשות:

✅ חיברת לדרייב שלך עם OAuth מאובטח  
✅ הורדת תמונות אמיתיות מאירוע  
✅ תייגת את כל התמונות בעזרת LLM Vision  
✅ חישבת ציוני אלבום לכל תמונה  
✅ תכננת מבנה אלבום באמצעות LLM Planner  
✅ בחרת תמונות מיטביות עם constraints  
✅ שיבצת תמונות ל-slots אופטימלית  
✅ יצרת IDML סופי עם תמונות מוטמעות  
✅ העלית את התוצאות חזרה לדרייב  

### 📥 קבצים שנוצרו:

</div>

In [ ]:
# === Cell 52 ===
# כל הקבצים שנוצרו
print("📁 קבצים בפרויקט:\n")
for file in sorted(OUTPUT_DIR.rglob("*")):
    if file.is_file():
        rel = file.relative_to(OUTPUT_DIR)
        size_kb = file.stat().st_size / 1024
        print(f"  {rel} ({size_kb:.1f} KB)")

print(f"\n📂 בדרייב: {album_folder_name}")

<!-- Cell 53 (Markdown) -->
<div dir=RTL>

## 🧪 ניסויים וכיול

חלק זה מאפשר לך לנסות שינויים ולראות את ההשפעה.

In [ ]:
# === Cell 54 ===
# 🧪 ניסוי 1: משקלים אחרים לציון האלבום
custom_weights = {
    'quality': 0.50,       # דגש חזק על איכות
    'people': 0.30,
    'uniqueness': 0.15,
    'emotional': 0.05,
}

for img in enriched:
    img.album_score = calculate_album_score(img, weights=custom_weights)

top_5 = sorted(enriched, key=lambda x: x.album_score, reverse=True)[:5]
print("🏆 5 המובילים עם משקלים חדשים:")
for img in top_5:
    print(f"  {img.filename} - {img.album_score:.1f}")

# שחזור למשקלים רגילים
for img in enriched:
    img.album_score = calculate_album_score(img)

In [ ]:
# === Cell 55 ===
# 🧪 ניסוי 2: אילוץ חזק של בת מצווה (70% במקום 40%)
strict = select_images_for_album(
    enriched, target_count=20,
    constraints={'min_batmitzvah_pct': 0.70, 'min_scene_diversity': 4, 'max_per_cluster': 2}
)

bm_ratio = sum(1 for img in strict if img.has_batmitzvah) / len(strict)
print(f"📊 אילוץ 70% בת מצווה:")
print(f"  נבחרו: {len(strict)} תמונות")
print(f"  אחוז בת מצווה בפועל: {bm_ratio:.0%}")

In [ ]:
# === Cell 56 ===
# 🧪 ניסוי 3: השוואת בחירה - מה נבחר מכל סצנה?
all_counts = Counter(img.scene_type for img in enriched)
sel_counts = Counter(img.scene_type for img in selected)

print("📊 יחס בחירה לפי סצנה:\n")
print(f"{'סצנה':<30} {'זמין':<6} {'נבחר':<6} {'אחוז':<6}")
print("-" * 55)
for scene, total in all_counts.most_common():
    chosen = sel_counts.get(scene, 0)
    pct = (chosen / total * 100) if total else 0
    print(f"{scene:<30} {total:<6} {chosen:<6} {pct:<5.0f}%")

<!-- Cell 57 (Markdown) -->
<div dir=RTL>

## 💡 הצעדים הבאים

### לאירוע מלא (2,000 תמונות):

1. שנה את `MAX_IMAGES_FOR_TEST` ל-`None` 
2. הרץ עם GPU של Colab (Runtime → Change runtime type → T4)
3. זמן צפוי לתיוג: ~30-60 דקות
4. עלות LLM: ~$2-3 (Gemini Flash) או ~$30 (Claude Sonnet)

### שדרוגים אפשריים:

- 🔧 **InsightFace** במקום OpenCV - זיהוי פנים מדויק יותר
- 🔧 **CLIP** כסינון מקדים לפני LLM Vision (חיסכון כספי)
- 🔧 **Face clustering** - לזהות **מי** האנשים, לא רק תפקידים
- 🔧 **תבניות שלך** - תוכל להחליף את התבנית האוטומטית
- 🔧 **CMYK להדפסה** - עם ICC profile של בית הדפוס שלך
- 🔧 **16 דפים מלאים** - לולאה על כל הפרקים, לא רק דף אחד

</div>